# 06 Spatial Distribution Analysis

This notebook handles the Subagent 06 spatial distribution analysis. Inputs are limited to the new master data `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` (expected `N=9222`) and three auxiliary files:

- `data/recipient_org_english_name_map.csv`
- `data/recipient_org_geo_mapping.csv`
- `data/china_province_basemap.geojson`

The output logic is retained for a later Fig5/table/log worker rerun:

- `output/figures/Fig5_spatial_distribution.svg/pdf/tiff/png`
- `output/tables/06_spatial_distribution.csv`
- `output/logs/06_spatial_text.md`

Spatial scope note: structured study-place fields (`knowledge_prod_place`, `research_object_place`, `beneficiary_city`) may still be empty in the new master table; this notebook's maps, city summaries, and regional summaries are based on recipient-organization geography (inferred recipient-organization location), not study-area geography.

Note: all visible figure text is in English; code comments document the reproducible rules for each step.


## Figure contract

- Core conclusion: Recipient-organization locations reveal an institution-level geography for the `N=9222` deduplicated corpus, and the leading funded recipient organizations differ across the seven inferred province regions.
- Evidence chain: panel a maps unique recipient institutions whose city can be inferred from `recipient_org`, validates the lon/lat-to-Web-Mercator coordinates before plotting, adds a scale bar, and labels the top 10 inferred cities with red callouts; panel b ranks the leading inferred cities by unique institutions, keeps bars colored by province region, and marks the top 10 city labels in red; panel c lists the top 8 recipient organizations by record count within each inferred province region, with four region blocks in the first row and three in the second row, using narrower blocks to remove empty right-side space.
- Archetype: asymmetric mixed-modality figure with spatial and city-ranking panels placed side by side in the first row plus a lower regional recipient-organization matrix.
- Backend: Python/matplotlib/geopandas only.
- Export contract: white page background, runtime Mapbox WMTS/XYZ Web Mercator background for panel a, editable SVG/PDF text, source-data CSV, and SVG/PDF/TIFF/PNG outputs.


In [1]:
# Import core libraries.
# pandas reads and summarizes CSV files; numpy handles numeric scaling; matplotlib/geopandas draw static maps and ranking charts.
from pathlib import Path
from datetime import datetime
from io import BytesIO
import hashlib
import math
import os
import re
import textwrap
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from PIL import Image
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.patheffects as path_effects
from pyproj import Transformer
from pyproj import Transformer
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from pyproj import CRS, Transformer

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a fallback.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

try:
    import geopandas as gpd
except Exception:
    gpd = None

try:
    import cartopy
except Exception:
    cartopy = None

try:
    import mercantile
except Exception:
    mercantile = None

# Set the project root so the notebook can find inputs and outputs when run from the project root or code/ subdirectory.
MAIN_DATA_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
EXPECTED_MAIN_RECORDS = 9222
CWD = Path.cwd()
ROOT = CWD if (CWD / "data" / MAIN_DATA_FILENAME).exists() else CWD.parent

DATA_PATH = ROOT / "data" / MAIN_DATA_FILENAME
ORG_ENGLISH_NAME_MAP_PATH = ROOT / "data" / "recipient_org_english_name_map.csv"
ORG_GEO_MAPPING_PATH = ROOT / "data" / "recipient_org_geo_mapping.csv"
CHINA_BASEMAP_PATH = ROOT / "data" / "china_province_basemap.geojson"
ALLOWED_INPUT_PATHS = [DATA_PATH, ORG_ENGLISH_NAME_MAP_PATH, ORG_GEO_MAPPING_PATH, CHINA_BASEMAP_PATH]
missing_input_paths = [str(path) for path in ALLOWED_INPUT_PATHS if not path.exists()]
if missing_input_paths:
    raise FileNotFoundError(f"Missing allowed input files: {missing_input_paths}")

FIG_BASE = ROOT / "output" / "figures" / "Fig5_spatial_distribution"
TABLE_PATH = ROOT / "output" / "tables" / "06_spatial_distribution.csv"
LOG_PATH = ROOT / "output" / "logs" / "06_spatial_text.md"
MAPBOX_TOKEN_ENV_VAR = "MAPBOX_ACCESS_TOKEN"
MAPBOX_WMTS_URL_ENV_VAR = "FIG5_MAPBOX_WMTS_URL"
MAPBOX_TILE_TEMPLATE_ENV_VAR = "FIG5_MAPBOX_TILE_URL_TEMPLATE"
MAPBOX_STYLE_OWNER_ENV_VAR = "FIG5_MAPBOX_STYLE_OWNER"
MAPBOX_STYLE_ID_ENV_VAR = "FIG5_MAPBOX_STYLE_ID"
MAPBOX_ACCESS_TOKEN_PLACEHOLDER = "{access_token}"
MAPBOX_CACHE_DIR = ROOT / "output" / "cache" / "fig5_mapbox"
MAPBOX_TILE_ZOOM = 4
MAPBOX_CACHE_WRITE_ENABLED = False

# Create only the output directories owned by this subagent; Mapbox uses runtime URLs only and does not write tiles/cache.
FIG_BASE.parent.mkdir(parents=True, exist_ok=True)
TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Use a China-region Albers projection; institution-point jitter is measured in meters, so the projection must be fixed before plotting.
ALBERS = CRS.from_proj4(
    "+proj=aea +lon_0=110 +lat_0=0 +lat_1=25 +lat_2=47 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)
WEB_MERCATOR = CRS.from_epsg(3857)
PROJECTOR = Transformer.from_crs("EPSG:4326", ALBERS, always_xy=True)
WEB_MERCATOR_PROJECTOR = Transformer.from_crs("EPSG:4326", WEB_MERCATOR, always_xy=True)
WEB_MERCATOR_TO_LONLAT = Transformer.from_crs(WEB_MERCATOR, "EPSG:4326", always_xy=True)

# Set journal-figure Times New Roman fonts and vector export options to keep SVG/PDF text as editable as possible.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.7,
    "legend.frameon": False,
})

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")


In [2]:
# Read the new master data, fixed English institution-name map, and institution geography map, then check field integrity.
# This notebook reads only the four allowed input files under data/ and does not write back to any file under data/.
df = pd.read_csv(DATA_PATH)

STRUCTURED_PLACE_FIELDS = ["knowledge_prod_place", "research_object_place", "beneficiary_city"]
REQUIRED_COLUMNS = [
    "award_id",
    "award_year",
    "recipient_org",
    *STRUCTURED_PLACE_FIELDS,
]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Main data is missing required columns: {missing_columns}")

n_records = len(df)
if n_records == 0:
    raise ValueError("Main data is empty; spatial distribution analysis cannot proceed.")
if n_records != EXPECTED_MAIN_RECORDS:
    raise ValueError(f"Main data should contain N={EXPECTED_MAIN_RECORDS}; got N={n_records}: {DATA_PATH}")

# Standardize missing-value handling: treat NaN, empty strings, and common null text values as missing.
def is_nonempty(series: pd.Series) -> pd.Series:
    text = series.fillna("").astype(str).str.strip()
    return text.ne("") & ~text.str.lower().isin({"nan", "none", "null", "na"})


for field in ["recipient_org", *STRUCTURED_PLACE_FIELDS]:
    df[field] = df[field].fillna("").astype(str).str.strip()

# Use a fixed English institution-name map to avoid online name lookups whenever Fig. 5 runs.
# The new master table has more institutions than the old map covers; missing English names no longer stop the statistics and are reported explicitly in tables and logs.
ORG_MAP_COLUMNS = ["recipient_org", "english_name", "source_type", "source_url", "verification_status", "review_note"]
ORG_ENGLISH_NAME_MAP_DF = pd.read_csv(ORG_ENGLISH_NAME_MAP_PATH)
missing_map_columns = [col for col in ORG_MAP_COLUMNS if col not in ORG_ENGLISH_NAME_MAP_DF.columns]
if missing_map_columns:
    raise ValueError(f"English institution-name map is missing columns: {missing_map_columns}")

ORG_ENGLISH_NAME_MAP_DF = ORG_ENGLISH_NAME_MAP_DF[ORG_MAP_COLUMNS].copy()
ORG_ENGLISH_NAME_MAP_DF["recipient_org"] = ORG_ENGLISH_NAME_MAP_DF["recipient_org"].fillna("").astype(str).str.strip()
ORG_ENGLISH_NAME_MAP_DF["english_name"] = ORG_ENGLISH_NAME_MAP_DF["english_name"].fillna("").astype(str).str.strip()
ORG_ENGLISH_NAME_MAP_DF = ORG_ENGLISH_NAME_MAP_DF.loc[ORG_ENGLISH_NAME_MAP_DF["recipient_org"].ne("")].copy()

duplicate_orgs = sorted(ORG_ENGLISH_NAME_MAP_DF.loc[ORG_ENGLISH_NAME_MAP_DF["recipient_org"].duplicated(), "recipient_org"].unique())
if duplicate_orgs:
    raise ValueError(f"English institution-name map contains duplicate institutions: {duplicate_orgs}")

empty_english_orgs = sorted(ORG_ENGLISH_NAME_MAP_DF.loc[ORG_ENGLISH_NAME_MAP_DF["english_name"].eq(""), "recipient_org"].unique())
if empty_english_orgs:
    raise ValueError(f"English institution-name map contains empty English names: {empty_english_orgs}")

# Use a fixed institution geography map. Preserve all mapping-table fields and keep original audit information in output tables with the geo_mapping_ prefix.
ORG_GEO_REQUIRED_COLUMNS = [
    "recipient_org_raw",
    "city",
    "province",
    "lon_wgs84",
    "lat_wgs84",
    "source_type",
    "confidence",
    "needs_review",
]
ORG_GEO_MAPPING_DF = pd.read_csv(ORG_GEO_MAPPING_PATH)
missing_geo_columns = [col for col in ORG_GEO_REQUIRED_COLUMNS if col not in ORG_GEO_MAPPING_DF.columns]
if missing_geo_columns:
    raise ValueError(f"Institution geography map is missing columns: {missing_geo_columns}")

ORG_GEO_MAPPING_DF = ORG_GEO_MAPPING_DF.copy()
GEO_MAPPING_SOURCE_COLUMNS = list(ORG_GEO_MAPPING_DF.columns)
for col in ["recipient_org_raw", "city", "province", "source_type", "source_url", "review_note"]:
    if col in ORG_GEO_MAPPING_DF.columns:
        ORG_GEO_MAPPING_DF[col] = ORG_GEO_MAPPING_DF[col].fillna("").astype(str).str.strip()
for col in ["lon_wgs84", "lat_wgs84", "confidence"]:
    ORG_GEO_MAPPING_DF[col] = pd.to_numeric(ORG_GEO_MAPPING_DF[col], errors="coerce")

def normalize_needs_review(value):
    text = "" if pd.isna(value) else str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n", ""}:
        return False
    raise ValueError(f"The needs_review field contains an unparseable boolean value: {value!r}")


ORG_GEO_MAPPING_DF["needs_review"] = ORG_GEO_MAPPING_DF["needs_review"].map(normalize_needs_review)
ORG_GEO_MAPPING_DF = ORG_GEO_MAPPING_DF.loc[ORG_GEO_MAPPING_DF["recipient_org_raw"].ne("")].copy()

invalid_geo_required = {}
for col in ["lon_wgs84", "lat_wgs84", "confidence"]:
    bad = ORG_GEO_MAPPING_DF[col].isna()
    if bad.any():
        invalid_geo_required[col] = ORG_GEO_MAPPING_DF.loc[bad, "recipient_org_raw"].head(10).tolist()
for col in ["recipient_org_raw", "city", "province", "source_type"]:
    bad = ~is_nonempty(ORG_GEO_MAPPING_DF[col])
    if bad.any():
        invalid_geo_required[col] = ORG_GEO_MAPPING_DF.loc[bad, "recipient_org_raw"].head(10).tolist()
if invalid_geo_required:
    raise ValueError(f"Institution geography map has missing or invalid required fields: {invalid_geo_required}")

if not ORG_GEO_MAPPING_DF["lon_wgs84"].between(73, 136).all() or not ORG_GEO_MAPPING_DF["lat_wgs84"].between(18, 54).all():
    bad_geo = ORG_GEO_MAPPING_DF.loc[
        ~ORG_GEO_MAPPING_DF["lon_wgs84"].between(73, 136) | ~ORG_GEO_MAPPING_DF["lat_wgs84"].between(18, 54),
        ["recipient_org_raw", "city", "province", "lon_wgs84", "lat_wgs84"],
    ].head(10).to_dict("records")
    raise ValueError(f"Institution geography map contains lon/lat values outside China: {bad_geo}")

duplicate_geo_orgs = sorted(ORG_GEO_MAPPING_DF.loc[ORG_GEO_MAPPING_DF["recipient_org_raw"].duplicated(), "recipient_org_raw"].unique())
if duplicate_geo_orgs:
    raise ValueError(f"Institution geography map contains duplicate institutions: {duplicate_geo_orgs}")

source_orgs = sorted(df.loc[is_nonempty(df["recipient_org"]), "recipient_org"].unique())
n_unique_institutions = len(source_orgs)
missing_org_name_map = sorted(set(source_orgs) - set(ORG_ENGLISH_NAME_MAP_DF["recipient_org"]))
missing_org_geo_map = sorted(set(source_orgs) - set(ORG_GEO_MAPPING_DF["recipient_org_raw"]))
extra_org_geo_map = sorted(set(ORG_GEO_MAPPING_DF["recipient_org_raw"]) - set(source_orgs))

ORG_GEO_MAPPING_LOOKUP = {
    row["recipient_org_raw"]: row.to_dict()
    for _, row in ORG_GEO_MAPPING_DF.iterrows()
}
GEO_MAPPING_AUDIT_COLUMNS = [f"geo_mapping_{col}" for col in GEO_MAPPING_SOURCE_COLUMNS]

ORG_ENGLISH_NAME_MAP = dict(zip(ORG_ENGLISH_NAME_MAP_DF["recipient_org"], ORG_ENGLISH_NAME_MAP_DF["english_name"]))
df["recipient_org_en"] = df["recipient_org"].map(ORG_ENGLISH_NAME_MAP).fillna("").astype(str).str.strip()
df["recipient_org_en_mapped"] = df["recipient_org_en"].ne("")
df["institution_display_name"] = df["recipient_org_en"].where(df["recipient_org_en_mapped"], df["recipient_org"])
df["recipient_org_name_source"] = np.where(df["recipient_org_en_mapped"], "english_name_map", "main_table_recipient_org")

mapped_orgs = sorted(df.loc[df["recipient_org_en_mapped"], "recipient_org"].unique())
n_mapped_english_institutions = len(mapped_orgs)
n_unmapped_english_institutions = n_unique_institutions - n_mapped_english_institutions
n_mapped_english_records = int(df["recipient_org_en_mapped"].sum())

FIELD_LABELS = {
    "recipient_org": "recipient organization",
    "recipient_org_english_map": "recipient organization English-name map",
    "knowledge_prod_place": "knowledge-production place",
    "research_object_place": "research-object place",
    "beneficiary_city": "beneficiary city",
    "inferred_institution_city": "inferred recipient-organization city",
    "inferred_institution_province": "inferred recipient-organization province",
}

# First record raw-field and English-name mapping coverage; later logs state whether structured study-place fields are usable.
coverage_rows = []
for field in ["recipient_org", *STRUCTURED_PLACE_FIELDS]:
    nonempty = is_nonempty(df[field])
    coverage_rows.append({
        "field": field,
        "field_label": FIELD_LABELS[field],
        "n_nonempty": int(nonempty.sum()),
        "coverage_pct": float(nonempty.mean() * 100),
    })
coverage_rows.append({
    "field": "recipient_org_english_map",
    "field_label": FIELD_LABELS["recipient_org_english_map"],
    "n_nonempty": n_mapped_english_records,
    "coverage_pct": float(n_mapped_english_records / n_records * 100),
})
coverage_df = pd.DataFrame(coverage_rows)

coverage_df, {
    "source_file": str(DATA_PATH.relative_to(ROOT)),
    "geo_mapping_file": str(ORG_GEO_MAPPING_PATH.relative_to(ROOT)),
    "n_records": n_records,
    "n_unique_recipient_orgs": n_unique_institutions,
    "n_mapped_english_institutions": n_mapped_english_institutions,
    "n_unmapped_english_institutions": n_unmapped_english_institutions,
    "n_unmapped_english_institution_sample": missing_org_name_map[:20],
    "n_geo_mapping_rows": len(ORG_GEO_MAPPING_DF),
    "n_missing_geo_mapping_institutions": len(missing_org_geo_map),
    "n_extra_geo_mapping_rows": len(extra_org_geo_map),
    "geo_mapping_extra_columns_preserved": [col for col in GEO_MAPPING_SOURCE_COLUMNS if col not in ORG_GEO_REQUIRED_COLUMNS],
}


(                       field                              field_label  \
 0              recipient_org                   recipient organization   
 1       knowledge_prod_place               knowledge-production place   
 2      research_object_place                    research-object place   
 3           beneficiary_city                         beneficiary city   
 4  recipient_org_english_map  recipient organization English-name map   
 
    n_nonempty  coverage_pct  
 0        9222    100.000000  
 1           0      0.000000  
 2           0      0.000000  
 3           0      0.000000  
 4        5073     55.009759  ,
 {'source_file': 'data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv',
  'geo_mapping_file': 'data/recipient_org_geo_mapping.csv',
  'n_records': 9222,
  'n_unique_recipient_orgs': 855,
  'n_mapped_english_institutions': 166,
  'n_unmapped_english_institutions': 689,
  'n_unmapped_english_institution_sample': ['三峡大学',
   '上海中医药大学',
   '上海宇航系统工程研究所',
   '上海工程技术大学',
   '上海市疾病预防控

In [3]:
# Build the institution-place inference dictionary.
# Rules:
# 1. Structured study-place fields may be empty; this task infers recipient-organization locations only from recipient_org;
# 2. English city names are used in figure text, and English province names are used in summaries and maps;
# 3. lon/lat values are approximate city centers for recipient organizations, used only for institution-point visualization and not as study areas or exact addresses;
# 4. terms include two kinds of clues: institution-specific names and city/province terms that appear directly in institution text.
CITY_META = {
    "Beijing": {"province": "Beijing", "lon": 116.4074, "lat": 39.9042, "terms": ["中国矿业大学（北京）", "中国石油大学（北京）", "中国地质大学（北京）", "中国科学院地理科学与资源研究所", "中国科学院生态环境研究中心", "中国科学院空间应用工程与技术中心", "中国科学院大学", "中国科学院软件研究所", "中国科学院科技战略咨询研究院", "中国农业科学院", "中国资源卫星应用中心", "中国建筑科学研究院", "中国地震局地质研究所", "生态环境部环境规划院", "国务院发展研究中心", "华北电力大学", "清华大学", "北京大学", "北京", "首都"]},
    "Shanghai": {"province": "Shanghai", "lon": 121.4737, "lat": 31.2304, "terms": ["中国科学院上海高等研究院", "中国人民解放军第二军医大学", "华东师范大学", "同济大学", "上海"]},
    "Guangzhou": {"province": "Guangdong", "lon": 113.2644, "lat": 23.1291, "terms": ["中国科学院华南植物园", "华南理工大学", "华南师范大学", "华南农业大学", "广东财经大学", "广东工业大学", "广东省生态气象中心", "中山大学", "广州"]},
    "Shenzhen": {"province": "Guangdong", "lon": 114.0579, "lat": 22.5431, "terms": ["中国科学院深圳先进技术研究院", "香港大学深圳研究院", "北京大学深圳研究生院", "南方科技大学", "深圳"]},
    "Wuhan": {"province": "Hubei", "lon": 114.3054, "lat": 30.5928, "terms": ["中国地质大学（武汉）", "华中农业大学", "华中师范大学", "华中科技大学", "湖北大学", "武汉"]},
    "Nanjing": {"province": "Jiangsu", "lon": 118.7969, "lat": 32.0603, "terms": ["中国地质调查局南京地质调查中心", "江苏省气象科学研究所", "江苏第二师范学院", "东南大学", "河海大学", "南京"]},
    "Tianjin": {"province": "Tianjin", "lon": 117.2000, "lat": 39.1333, "terms": ["河北工业大学", "南开大学", "天津"]},
    "Chongqing": {"province": "Chongqing", "lon": 106.5516, "lat": 29.5630, "terms": ["中国科学院重庆绿色智能技术研究院", "重庆"]},
    "Hangzhou": {"province": "Zhejiang", "lon": 120.1551, "lat": 30.2741, "terms": ["香港大学浙江科学技术研究院", "浙江大学", "浙江工业大学", "浙江理工大学", "浙江工商大学", "杭州"]},
    "Harbin": {"province": "Heilongjiang", "lon": 126.5349, "lat": 45.8038, "terms": ["中国地震局工程力学研究所", "哈尔滨"]},
    "Shenyang": {"province": "Liaoning", "lon": 123.4315, "lat": 41.8057, "terms": ["中国科学院沈阳应用生态研究所", "中国气象局沈阳大气环境研究所", "沈阳"]},
    "Urumqi": {"province": "Xinjiang", "lon": 87.6168, "lat": 43.8256, "terms": ["新疆师范大学", "新疆农业大学", "乌鲁木齐"]},
    "Nanning": {"province": "Guangxi", "lon": 108.3669, "lat": 22.8170, "terms": ["南宁师范大学", "广西财经学院", "广西大学", "南宁"]},
    "Xiamen": {"province": "Fujian", "lon": 118.0894, "lat": 24.4798, "terms": ["中国科学院城市环境研究所", "清华海峡研究院（厦门）", "厦门"]},
    "Xi'an": {"province": "Shaanxi", "lon": 108.9398, "lat": 34.3416, "terms": ["陕西师范大学", "西安电子科技大学", "西安理工大学", "西安工程大学", "西北大学", "长安大学", "西安"]},
    "Guilin": {"province": "Guangxi", "lon": 110.2900, "lat": 25.2736, "terms": ["桂林"]},
    "Jinan": {"province": "Shandong", "lon": 117.1201, "lat": 36.6512, "terms": ["山东建筑大学", "山东师范大学", "济南"]},
    "Kunming": {"province": "Yunnan", "lon": 102.8329, "lat": 24.8801, "terms": ["西南林业大学", "云南大学", "云南师范大学", "昆明"]},
    "Fuzhou": {"province": "Fujian", "lon": 119.2965, "lat": 26.0745, "terms": ["福建理工大学", "福州"]},
    "Changsha": {"province": "Hunan", "lon": 112.9388, "lat": 28.2282, "terms": ["湖南第一师范学院", "湖南工商大学", "湖南师范大学", "中南大学", "湖南大学", "长沙"]},
    "Lanzhou": {"province": "Gansu", "lon": 103.8343, "lat": 36.0611, "terms": ["西北师范大学", "兰州"]},
    "Nanchang": {"province": "Jiangxi", "lon": 115.8582, "lat": 28.6820, "terms": ["江西财经大学", "南昌"]},
    "Chengdu": {"province": "Sichuan", "lon": 104.0665, "lat": 30.5723, "terms": ["中国科学院、水利部成都山地灾害与环境研究所", "成都信息工程大学", "西南交通大学", "四川大学", "成都"]},
    "Qingdao": {"province": "Shandong", "lon": 120.3826, "lat": 36.0671, "terms": ["中国石油大学（华东）", "青岛"]},
    "Hefei": {"province": "Anhui", "lon": 117.2272, "lat": 31.8206, "terms": ["合肥"]},
    "Guiyang": {"province": "Guizhou", "lon": 106.6302, "lat": 26.6470, "terms": ["贵州师范学院", "贵阳"]},
    "Shijiazhuang": {"province": "Hebei", "lon": 114.5149, "lat": 38.0428, "terms": ["河北地质大学", "石家庄"]},
    "Xuzhou": {"province": "Jiangsu", "lon": 117.2841, "lat": 34.2058, "terms": ["江苏师范大学", "中国矿业大学", "徐州"]},
    "Suzhou": {"province": "Jiangsu", "lon": 120.5853, "lat": 31.2989, "terms": ["西交利物浦大学", "苏州"]},
    "Zhenjiang": {"province": "Jiangsu", "lon": 119.4250, "lat": 32.1896, "terms": ["江苏大学", "镇江"]},
    "Quanzhou": {"province": "Fujian", "lon": 118.6759, "lat": 24.8741, "terms": ["华侨大学", "泉州"]},
    "Ningbo": {"province": "Zhejiang", "lon": 121.5503, "lat": 29.8746, "terms": ["宁波"]},
    "Kaifeng": {"province": "Henan", "lon": 114.3076, "lat": 34.7973, "terms": ["河南大学", "开封"]},
    "Zhengzhou": {"province": "Henan", "lon": 113.6254, "lat": 34.7466, "terms": ["河南财经政法大学", "郑州"]},
    "Jiaozuo": {"province": "Henan", "lon": 113.2418, "lat": 35.2159, "terms": ["河南理工大学", "焦作"]},
    "Hohhot": {"province": "Inner Mongolia", "lon": 111.7492, "lat": 40.8426, "terms": ["内蒙古大学", "内蒙古师范大学", "呼和浩特"]},
    "Changchun": {"province": "Jilin", "lon": 125.3235, "lat": 43.8171, "terms": ["东北师范大学", "长春"]},
    "Yanji": {"province": "Jilin", "lon": 129.5089, "lat": 42.8913, "terms": ["延边大学", "延边"]},
    "Haikou": {"province": "Hainan", "lon": 110.1983, "lat": 20.0444, "terms": ["海南师范大学", "海口"]},
    "Xining": {"province": "Qinghai", "lon": 101.7782, "lat": 36.6171, "terms": ["青海大学", "西宁"]},
    "Taiyuan": {"province": "Shanxi", "lon": 112.5489, "lat": 37.8706, "terms": ["山西财经大学", "太原"]},
    "Ma'anshan": {"province": "Anhui", "lon": 118.5061, "lat": 31.6705, "terms": ["安徽工业大学", "马鞍山"]},
    "Yantai": {"province": "Shandong", "lon": 121.4479, "lat": 37.4638, "terms": ["烟台"]},
    "Hengyang": {"province": "Hunan", "lon": 112.5720, "lat": 26.8942, "terms": ["衡阳"]},
}

# Direct geographic-term clues distinguish institution-name lookup from explicit place mentions in institution text.
EXPLICIT_GEO_CUES = {
    "北京", "上海", "天津", "重庆", "武汉", "南京", "广州", "深圳", "杭州", "哈尔滨", "沈阳", "乌鲁木齐",
    "南宁", "厦门", "西安", "桂林", "济南", "昆明", "福州", "长沙", "兰州", "南昌", "成都", "青岛",
    "合肥", "贵阳", "石家庄", "徐州", "苏州", "镇江", "泉州", "宁波", "开封", "郑州", "焦作",
    "呼和浩特", "长春", "延边", "海口", "西宁", "太原", "马鞍山", "烟台", "衡阳",
    "新疆", "广西", "山东", "浙江", "河南", "江苏", "广东", "福建", "湖南", "湖北", "陕西", "甘肃",
    "云南", "四川", "安徽", "贵州", "河北", "内蒙古", "吉林", "海南", "青海", "山西", "辽宁", "黑龙江", "江西"
}

# Match all trigger terms in descending length so longer names are prioritized over shorter substrings.
TERM_TO_CITY = []
for city, meta in CITY_META.items():
    for term in meta["terms"]:
        TERM_TO_CITY.append((term, city))
TERM_TO_CITY = sorted(TERM_TO_CITY, key=lambda x: len(x[0]), reverse=True)


def _blank_geo_mapping_audit_fields() -> dict:
    return {column: np.nan for column in GEO_MAPPING_AUDIT_COLUMNS}


def _geo_mapping_value(mapping_row: dict, column: str, default=np.nan):
    value = mapping_row.get(column, default)
    if isinstance(value, str):
        return value.strip()
    return value


def _location_result_from_geo_mapping(org_text: str, mapping_row: dict) -> dict:
    result = _blank_geo_mapping_audit_fields()
    for source_col in GEO_MAPPING_SOURCE_COLUMNS:
        result[f"geo_mapping_{source_col}"] = _geo_mapping_value(mapping_row, source_col)
    result.update({
        "city": _geo_mapping_value(mapping_row, "city"),
        "province": _geo_mapping_value(mapping_row, "province"),
        "lon": float(_geo_mapping_value(mapping_row, "lon_wgs84")),
        "lat": float(_geo_mapping_value(mapping_row, "lat_wgs84")),
        "match_term": org_text,
        "location_source": "recipient_org_geo_mapping_exact",
        "source_type": _geo_mapping_value(mapping_row, "source_type"),
        "confidence": float(_geo_mapping_value(mapping_row, "confidence")),
        "needs_review": bool(_geo_mapping_value(mapping_row, "needs_review", False)),
        "source_url": _geo_mapping_value(mapping_row, "source_url"),
        "review_note": _geo_mapping_value(mapping_row, "review_note"),
    })
    return result


def _location_result_from_city_meta(city: str, match_term: str, source: str) -> dict:
    meta = CITY_META[city]
    result = _blank_geo_mapping_audit_fields()
    result.update({
        "city": city,
        "province": meta["province"],
        "lon": meta["lon"],
        "lat": meta["lat"],
        "match_term": match_term,
        "location_source": source,
        "source_type": f"city_meta_fallback_{source}",
        "confidence": np.nan,
        "needs_review": True,
        "source_url": np.nan,
        "review_note": "Fallback to notebook CITY_META because recipient_org_geo_mapping.csv had no exact recipient_org_raw match.",
    })
    return result


def infer_org_location(org: str) -> dict:
    """Infer city, province, and coordinates from institution text; exact matches in the institution geography map take priority."""
    org_text = "" if pd.isna(org) else str(org).strip()
    if org_text in ORG_GEO_MAPPING_LOOKUP:
        return _location_result_from_geo_mapping(org_text, ORG_GEO_MAPPING_LOOKUP[org_text])

    for term, city in TERM_TO_CITY:
        if term in org_text:
            explicit_hits = [cue for cue in EXPLICIT_GEO_CUES if cue in org_text]
            source = "recipient_org_text_place_cue" if explicit_hits else "recipient_org_institution_lookup"
            return _location_result_from_city_meta(city, term, source)

    result = _blank_geo_mapping_audit_fields()
    result.update({
        "city": np.nan,
        "province": np.nan,
        "lon": np.nan,
        "lat": np.nan,
        "match_term": np.nan,
        "location_source": "unresolved",
        "source_type": "unresolved",
        "confidence": np.nan,
        "needs_review": True,
        "source_url": np.nan,
        "review_note": "No exact recipient_org_geo_mapping.csv match and no CITY_META fallback rule matched.",
    })
    return result


In [4]:
# Infer recipient-organization location for each record.
# Note: these coordinates represent the recipient organization city, not the project study area, knowledge-production place, or beneficiary city.
GEO_LOCATION_OUTPUT_FIELDS = [
    "source_type",
    "confidence",
    "needs_review",
    "source_url",
    "review_note",
    *GEO_MAPPING_AUDIT_COLUMNS,
]

location_records = []
for idx, row in df.iterrows():
    inferred = infer_org_location(row["recipient_org"])
    record = {
        "award_id": row["award_id"],
        "award_year": row["award_year"],
        "recipient_org": row["recipient_org"],
        "recipient_org_en": row["recipient_org_en"],
        "recipient_org_en_mapped": bool(row["recipient_org_en_mapped"]),
        "institution_display_name": row["institution_display_name"],
        "recipient_org_name_source": row["recipient_org_name_source"],
        "inferred_city": inferred["city"],
        "inferred_province": inferred["province"],
        "lon": inferred["lon"],
        "lat": inferred["lat"],
        "match_term": inferred["match_term"],
        "location_source": inferred["location_source"],
    }
    for field in GEO_LOCATION_OUTPUT_FIELDS:
        record[field] = inferred.get(field, np.nan)
    location_records.append(record)

record_locations = pd.DataFrame(location_records)

# If future data leaves any institution unresolved, stop immediately; Fig. 5 panel a must cover all unique recipient organizations.
city_nonempty = record_locations["inferred_city"].notna()
province_nonempty = record_locations["inferred_province"].notna()
coverage_df = pd.concat([
    coverage_df,
    pd.DataFrame([
        {"field": "inferred_institution_city", "field_label": FIELD_LABELS["inferred_institution_city"], "n_nonempty": int(city_nonempty.sum()), "coverage_pct": float(city_nonempty.mean() * 100)},
        {"field": "inferred_institution_province", "field_label": FIELD_LABELS["inferred_institution_province"], "n_nonempty": int(province_nonempty.sum()), "coverage_pct": float(province_nonempty.mean() * 100)},
    ])
], ignore_index=True)

unresolved_orgs = (
    record_locations.loc[record_locations["inferred_city"].isna(), "recipient_org"]
    .value_counts()
    .rename_axis("recipient_org")
    .reset_index(name="n_records")
)
if not unresolved_orgs.empty:
    raise ValueError(f"Unresolved recipient_org values remain; update recipient_org_geo_mapping.csv or CITY_META: {unresolved_orgs.head(20).to_dict('records')}")

coverage_df


,field,field_label,n_nonempty,coverage_pct
0,recipient_org,recipient organization,9222,100.000000
1,knowledge_prod_place,knowledge-production place,0,0.000000
2,research_object_place,research-object place,0,0.000000
3,beneficiary_city,beneficiary city,0,0.000000
4,recipient_org_english_map,recipient organization English-name map,5073,55.009759
5,inferred_institution_city,inferred recipient-organization city,9222,100.000000
6,inferred_institution_province,inferred recipient-organization province,9222,100.000000


In [5]:
# Generate institution-level, institution-point, institution audit-index, city, and source-type aggregate distributions.
# Key scope: the Fig. 5 map and city bar chart count unique recipient organizations; n_records is retained only as an auxiliary field.
def join_award_ids(values):
    return ";".join(sorted(values.dropna().astype(str).unique()))


def project_xy(table, lon_col="lon", lat_col="lat"):
    x = np.full(len(table), np.nan, dtype=float)
    y = np.full(len(table), np.nan, dtype=float)
    valid = table[lon_col].notna() & table[lat_col].notna()
    if valid.any():
        projected_x, projected_y = PROJECTOR.transform(
            table.loc[valid, lon_col].astype(float).to_numpy(),
            table.loc[valid, lat_col].astype(float).to_numpy(),
        )
        x[valid.to_numpy()] = projected_x
        y[valid.to_numpy()] = projected_y
    return x, y


def project_web_mercator_xy(table, lon_col="lon", lat_col="lat"):
    x = np.full(len(table), np.nan, dtype=float)
    y = np.full(len(table), np.nan, dtype=float)
    valid = table[lon_col].notna() & table[lat_col].notna()
    if valid.any():
        projected_x, projected_y = WEB_MERCATOR_PROJECTOR.transform(
            table.loc[valid, lon_col].astype(float).to_numpy(),
            table.loc[valid, lat_col].astype(float).to_numpy(),
        )
        x[valid.to_numpy()] = projected_x
        y[valid.to_numpy()] = projected_y
    return x, y


def stable_hash_hex(*parts):
    key = "||".join("" if pd.isna(part) else str(part) for part in parts)
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]


def stable_hash_unit(*parts):
    return int(stable_hash_hex(*parts), 16) / float(16 ** 16 - 1)


institution_group_columns = [
    "recipient_org", "recipient_org_en", "recipient_org_en_mapped", "institution_display_name", "recipient_org_name_source",
    "inferred_city", "inferred_province", "lon", "lat", "match_term", "location_source",
]
for column in ["source_type", "confidence", "needs_review", "source_url", "review_note", *GEO_MAPPING_AUDIT_COLUMNS]:
    if column in record_locations.columns and column not in institution_group_columns:
        institution_group_columns.append(column)

institution_base = (
    record_locations
    .groupby(
        institution_group_columns,
        as_index=False,
        dropna=False,
    )
    .agg(
        n_records=("award_id", "count"),
        award_year_min=("award_year", "min"),
        award_year_max=("award_year", "max"),
        award_ids=("award_id", join_award_ids),
    )
)
REGION_ORDER = [
    "Northeast China",
    "North China",
    "East China",
    "Central China",
    "South China",
    "Southwest China",
    "Northwest China",
]
REGION_RANK = {region: rank for rank, region in enumerate(REGION_ORDER)}

PROVINCE_REGION_MAP = {
    "Liaoning": "Northeast China",
    "Jilin": "Northeast China",
    "Heilongjiang": "Northeast China",
    "Beijing": "North China",
    "Tianjin": "North China",
    "Hebei": "North China",
    "Shanxi": "North China",
    "Inner Mongolia": "North China",
    "Shanghai": "East China",
    "Jiangsu": "East China",
    "Zhejiang": "East China",
    "Anhui": "East China",
    "Fujian": "East China",
    "Jiangxi": "East China",
    "Shandong": "East China",
    "Taiwan": "East China",
    "Henan": "Central China",
    "Hubei": "Central China",
    "Hunan": "Central China",
    "Guangdong": "South China",
    "Guangxi": "South China",
    "Hainan": "South China",
    "Hong Kong": "South China",
    "Macau": "South China",
    "Chongqing": "Southwest China",
    "Sichuan": "Southwest China",
    "Guizhou": "Southwest China",
    "Yunnan": "Southwest China",
    "Tibet": "Southwest China",
    "Shaanxi": "Northwest China",
    "Gansu": "Northwest China",
    "Qinghai": "Northwest China",
    "Ningxia": "Northwest China",
    "Xinjiang": "Northwest China",
}
INSTITUTION_INDEX_PROVINCE_ORDER = [
    "Liaoning", "Jilin", "Heilongjiang",
    "Beijing", "Tianjin", "Hebei", "Shanxi", "Inner Mongolia",
    "Shanghai", "Jiangsu", "Zhejiang", "Anhui", "Fujian", "Jiangxi", "Shandong", "Taiwan",
    "Henan", "Hubei", "Hunan",
    "Guangdong", "Guangxi", "Hainan", "Hong Kong", "Macau",
    "Chongqing", "Sichuan", "Guizhou", "Yunnan", "Tibet",
    "Shaanxi", "Gansu", "Qinghai", "Ningxia", "Xinjiang",
]
INSTITUTION_INDEX_PROVINCE_RANK = {province: rank for rank, province in enumerate(INSTITUTION_INDEX_PROVINCE_ORDER)}
INSTITUTION_INDEX_COLOR_GROUP_COUNT = len(REGION_ORDER)

institution_base["institution_sort_name"] = institution_base["institution_display_name"].fillna("").astype(str).str.lower()
institution_base["region"] = institution_base["inferred_province"].map(PROVINCE_REGION_MAP)
missing_region_provinces = sorted(institution_base.loc[institution_base["region"].isna(), "inferred_province"].dropna().unique().tolist())
if missing_region_provinces:
    raise ValueError(f"Institution provinces are missing region mappings: {missing_region_provinces}")
unknown_region_rank = len(REGION_ORDER)
unknown_province_rank = len(INSTITUTION_INDEX_PROVINCE_ORDER)
institution_base["region_sort_rank"] = institution_base["region"].map(REGION_RANK).fillna(unknown_region_rank).astype(int)
institution_base["region_color_group"] = institution_base["region_sort_rank"]
institution_base["province_sort_rank"] = (
    institution_base["inferred_province"].map(INSTITUTION_INDEX_PROVINCE_RANK).fillna(unknown_province_rank).astype(int)
)
institution_base["province_color_group"] = institution_base["region_color_group"]
institution_base = institution_base.sort_values(
    ["region_sort_rank", "province_sort_rank", "inferred_province", "institution_sort_name", "recipient_org"],
    kind="mergesort",
    na_position="last",
).reset_index(drop=True)
institution_base["institution_id"] = np.arange(1, len(institution_base) + 1)
institution_base["institution_name"] = institution_base["institution_display_name"]
institution_base["institution_name_en"] = institution_base["recipient_org_en"].replace("", np.nan)
institution_base["year_range"] = np.where(
    institution_base["award_year_min"].eq(institution_base["award_year_max"]),
    institution_base["award_year_min"].astype(str),
    institution_base["award_year_min"].astype(str) + "-" + institution_base["award_year_max"].astype(str),
)

n_institutions = int(len(institution_base))
if n_institutions != n_unique_institutions:
    raise ValueError(f"Institution-level base table row-count mismatch: {n_institutions} != {n_unique_institutions}")

missing_institution_english = institution_base.loc[
    ~institution_base["recipient_org_en_mapped"],
    "recipient_org",
].tolist()

institution_base["city_projected_x"], institution_base["city_projected_y"] = project_xy(institution_base)
institution_base["city_web_mercator_x"], institution_base["city_web_mercator_y"] = project_web_mercator_xy(institution_base)

institution_points = institution_base.dropna(subset=["city_projected_x", "city_projected_y", "city_web_mercator_x", "city_web_mercator_y"]).copy()
institution_points["jitter_hash"] = institution_points.apply(
    lambda row: stable_hash_hex("institution-jitter", row["recipient_org"], row["institution_name"], row["inferred_city"]),
    axis=1,
)
institution_points = institution_points.sort_values(["inferred_city", "jitter_hash", "institution_name"], kind="mergesort").reset_index(drop=True)
institution_points["city_institution_count"] = institution_points.groupby("inferred_city")["institution_id"].transform("count")
institution_points["jitter_order"] = institution_points.groupby("inferred_city").cumcount()
city_phase = {
    city: stable_hash_unit("city-phase", city) * 2 * np.pi
    for city in institution_points["inferred_city"].dropna().unique()
}
golden_angle = np.pi * (3 - np.sqrt(5))
institution_points["jitter_max_radius_m"] = np.clip(
    9000 + 7000 * np.sqrt(institution_points["city_institution_count"].astype(float)),
    16000,
    50000,
)
institution_points["jitter_radius_m"] = np.where(
    institution_points["city_institution_count"].le(1),
    0.0,
    institution_points["jitter_max_radius_m"] * np.sqrt(
        (institution_points["jitter_order"].astype(float) + 0.5)
        / institution_points["city_institution_count"].astype(float)
    ),
)
institution_points["jitter_angle_rad"] = (
    institution_points["inferred_city"].map(city_phase).astype(float)
    + institution_points["jitter_order"].astype(float) * golden_angle
)
institution_points["jittered_x"] = institution_points["city_projected_x"] + institution_points["jitter_radius_m"] * np.cos(institution_points["jitter_angle_rad"])
institution_points["jittered_y"] = institution_points["city_projected_y"] + institution_points["jitter_radius_m"] * np.sin(institution_points["jitter_angle_rad"])
institution_points["jittered_web_mercator_x"] = institution_points["city_web_mercator_x"] + institution_points["jitter_radius_m"] * np.cos(institution_points["jitter_angle_rad"])
institution_points["jittered_web_mercator_y"] = institution_points["city_web_mercator_y"] + institution_points["jitter_radius_m"] * np.sin(institution_points["jitter_angle_rad"])
institution_points = institution_points.sort_values("institution_id").reset_index(drop=True)

INDEX_COLUMNS = 4
INDEX_WRAP_WIDTH = 46
INDEX_ROW_SPACING_UNITS = 0.34
INDEX_SPLIT_PENALTY_UNITS = 7.0


def wrap_index_name(name, width=INDEX_WRAP_WIDTH):
    wrapped = textwrap.wrap(str(name), width=width, break_long_words=False, break_on_hyphens=False)
    return "\\n".join(wrapped or [str(name)])


def balanced_index_breaks(line_units, n_columns=INDEX_COLUMNS):
    """Split the continuous institution audit index into columns with approximately equal visual height after line wrapping."""
    n = len(line_units)
    breaks = [0]
    start = 0
    for col in range(1, n_columns):
        remaining_cols = n_columns - col
        target = float(np.sum(line_units[start:])) / (remaining_cols + 1)
        best_stop = start + 1
        best_score = np.inf
        running = 0.0
        max_stop = n - remaining_cols
        for stop in range(start + 1, max_stop + 1):
            running += float(line_units[stop - 1])
            score = abs(running - target)
            if score <= best_score:
                best_score = score
                best_stop = stop
        breaks.append(best_stop)
        start = best_stop
    breaks.append(n)
    return breaks


def balanced_group_index_breaks(index_df, n_columns=INDEX_COLUMNS):
    """Balance visual column height while penalizing, but allowing, province splits."""
    n = len(index_df)
    line_units = (index_df["index_line_count"].clip(lower=1).astype(float) + INDEX_ROW_SPACING_UNITS).to_numpy()
    groups = []
    start = 0
    group_keys = ["region_color_group", "inferred_province"]
    for _, group in index_df.groupby(group_keys, sort=False, dropna=False):
        stop = start + len(group)
        units = float(line_units[start:stop].sum())
        groups.append({"start": start, "stop": stop, "visual_units": units})
        start = stop
    group_boundaries = {0, n}
    for group in groups:
        group_boundaries.add(group["start"])
        group_boundaries.add(group["stop"])
    if n < n_columns:
        return list(range(n + 1)) + [n] * (n_columns - n)

    target = float(line_units.sum()) / n_columns
    prefix = np.r_[0.0, np.cumsum(line_units)]
    score = np.full((n_columns + 1, n + 1), np.inf)
    previous = np.full((n_columns + 1, n + 1), -1, dtype=int)
    score[0, 0] = 0.0
    for col in range(1, n_columns + 1):
        min_end = col
        max_end = n - (n_columns - col)
        for stop in range(min_end, max_end + 1):
            for start_idx in range(col - 1, stop):
                column_units = prefix[stop] - prefix[start_idx]
                split_penalty = 0.0 if stop in group_boundaries else INDEX_SPLIT_PENALTY_UNITS
                candidate = score[col - 1, start_idx] + abs(column_units - target) + split_penalty
                if candidate < score[col, stop]:
                    score[col, stop] = candidate
                    previous[col, stop] = start_idx
    breaks = [n]
    cursor = n
    for col in range(n_columns, 0, -1):
        cursor = int(previous[col, cursor])
        if cursor < 0:
            return balanced_index_breaks(line_units, n_columns)
        breaks.append(cursor)
    return sorted(breaks)


institution_index = institution_base.sort_values("institution_id").copy()
institution_index["wrapped_institution_name"] = institution_index["institution_name"].map(wrap_index_name)
institution_index["index_line_count"] = institution_index["wrapped_institution_name"].str.count("\\n") + 1
index_breaks = balanced_group_index_breaks(institution_index, INDEX_COLUMNS)
institution_index["index_col"] = 0
institution_index["index_row"] = 0
for index_col, (start, stop) in enumerate(zip(index_breaks[:-1], index_breaks[1:]), start=1):
    row_positions = np.arange(1, stop - start + 1)
    institution_index.iloc[start:stop, institution_index.columns.get_loc("index_col")] = index_col
    institution_index.iloc[start:stop, institution_index.columns.get_loc("index_row")] = row_positions
index_column_summary = (
    institution_index
    .groupby("index_col", as_index=False)
    .agg(n_institutions=("institution_id", "count"), visual_units=("index_line_count", lambda x: float((x.clip(lower=1) + INDEX_ROW_SPACING_UNITS).sum())))
)
index_province_col_counts = institution_index.dropna(subset=["inferred_province"]).groupby("inferred_province")["index_col"].nunique()
index_provinces_split_across_columns = sorted(index_province_col_counts.loc[index_province_col_counts.gt(1)].index.tolist()) if not index_province_col_counts.empty else []
institution_index["index_display"] = institution_index.apply(
    lambda row: f"{int(row['institution_id']):03d}. {row['institution_name']}",
    axis=1,
)

city_counts = (
    institution_base.dropna(subset=["inferred_city"])
    .groupby(["inferred_city", "inferred_province"], as_index=False)
    .agg(
        unique_orgs=("institution_id", "count"),
        n_records=("n_records", "sum"),
        award_year_min=("award_year_min", "min"),
        award_year_max=("award_year_max", "max"),
        lon=("lon", "mean"),
        lat=("lat", "mean"),
    )
    .sort_values(["unique_orgs", "n_records", "inferred_city"], ascending=[False, False, True], kind="mergesort")
    .reset_index(drop=True)
)
city_counts["rank"] = np.arange(1, len(city_counts) + 1)
city_counts["share_institutions"] = city_counts["unique_orgs"] / n_institutions
city_counts["share_records"] = city_counts["n_records"] / n_records
city_counts["region"] = city_counts["inferred_province"].map(PROVINCE_REGION_MAP)
city_counts["city_projected_x"], city_counts["city_projected_y"] = project_xy(city_counts)
city_counts["city_web_mercator_x"], city_counts["city_web_mercator_y"] = project_web_mercator_xy(city_counts)

province_counts = (
    institution_base.dropna(subset=["inferred_province"])
    .groupby("inferred_province", as_index=False)
    .agg(
        unique_orgs=("institution_id", "count"),
        n_records=("n_records", "sum"),
        n_cities=("inferred_city", "nunique"),
    )
    .sort_values(["unique_orgs", "n_records", "inferred_province"], ascending=[False, False, True], kind="mergesort")
    .reset_index(drop=True)
)
province_counts["rank"] = np.arange(1, len(province_counts) + 1)
province_counts["share_institutions"] = province_counts["unique_orgs"] / n_institutions
province_counts["share_records"] = province_counts["n_records"] / n_records
province_counts["region"] = province_counts["inferred_province"].map(PROVINCE_REGION_MAP)

source_counts = (
    institution_base
    .groupby("location_source", as_index=False)
    .agg(unique_orgs=("institution_id", "count"), n_records=("n_records", "sum"))
    .sort_values(["unique_orgs", "n_records"], ascending=[False, False], kind="mergesort")
    .reset_index(drop=True)
)
source_counts["rank"] = np.arange(1, len(source_counts) + 1)
source_counts["share_institutions"] = source_counts["unique_orgs"] / n_institutions
source_counts["share_records"] = source_counts["n_records"] / n_records

review_counts = (
    institution_base
    .assign(needs_review_label=np.where(institution_base["needs_review"].astype(bool), "needs review", "review not required"))
    .groupby(["needs_review", "needs_review_label"], as_index=False)
    .agg(unique_orgs=("institution_id", "count"), n_records=("n_records", "sum"))
    .sort_values(["needs_review", "unique_orgs"], ascending=[False, False], kind="mergesort")
    .reset_index(drop=True)
)
review_counts["rank"] = np.arange(1, len(review_counts) + 1)
review_counts["share_institutions"] = review_counts["unique_orgs"] / n_institutions
review_counts["share_records"] = review_counts["n_records"] / n_records

english_mapping_counts = (
    institution_base
    .assign(mapping_status=np.where(institution_base["recipient_org_en_mapped"], "mapped English name", "unmapped; original recipient_org retained"))
    .groupby("mapping_status", as_index=False)
    .agg(unique_orgs=("institution_id", "count"), n_records=("n_records", "sum"))
    .sort_values(["unique_orgs", "n_records"], ascending=[False, False], kind="mergesort")
    .reset_index(drop=True)
)
english_mapping_counts["rank"] = np.arange(1, len(english_mapping_counts) + 1)
english_mapping_counts["share_institutions"] = english_mapping_counts["unique_orgs"] / n_institutions
english_mapping_counts["share_records"] = english_mapping_counts["n_records"] / n_records

city_concentration_points = [1, 3, 5, 10, 15, 20]
city_concentration = pd.DataFrame([
    {
        "top_k": k,
        "unique_orgs": int(city_counts.head(k)["unique_orgs"].sum()),
        "share_institutions": float(city_counts.head(k)["unique_orgs"].sum() / n_institutions),
        "n_records": int(city_counts.head(k)["n_records"].sum()),
        "share_records": float(city_counts.head(k)["n_records"].sum() / n_records),
        "label": f"Top {k}",
    }
    for k in city_concentration_points
    if k <= len(city_counts)
])

city_counts.head(15), institution_index.shape, source_counts, english_mapping_counts


(   inferred_city inferred_province  unique_orgs  n_records  award_year_min  \
 0        Beijing           Beijing          175       1929            2010   
 1        Nanjing           Jiangsu           36        739            2010   
 2       Shanghai          Shanghai           35        631            2011   
 3      Guangzhou         Guangdong           34        454            2010   
 4          Xi'an           Shaanxi           28        371            2010   
 5       Hangzhou          Zhejiang           23        252            2011   
 6          Wuhan             Hubei           21        558            2010   
 7        Tianjin           Tianjin           20        267            2011   
 8        Chengdu           Sichuan           17        246            2011   
 9      Zhengzhou             Henan           17         87            2011   
 10       Lanzhou             Gansu           16        229            2010   
 11      Shenyang          Liaoning           16    

In [6]:
# Export the spatial distribution table.
# One CSV stores field coverage, the institution base table, institution points, institution audit index, city/province distributions, and location-inference source distributions.
coverage_table = pd.DataFrame({
    "section": "field_coverage",
    "field": coverage_df["field"],
    "place_name": coverage_df["field_label"],
    "n_records": coverage_df["n_nonempty"].astype(int),
    "share_records": coverage_df["coverage_pct"] / 100,
    "note": "source-field and mapping coverage; structured study-place fields may be empty",
})

city_table = city_counts.assign(
    section="institution_city_distribution",
    field="recipient_org_inferred",
    place_name=city_counts["inferred_city"],
    city=city_counts["inferred_city"],
    province=city_counts["inferred_province"],
    n_institutions=city_counts["unique_orgs"],
    note="city inferred from recipient_org; ranked by unique institutions; not study-area geography",
)

province_table = province_counts.assign(
    section="institution_province_distribution",
    field="recipient_org_inferred",
    place_name=province_counts["inferred_province"],
    province=province_counts["inferred_province"],
    n_institutions=province_counts["unique_orgs"],
    note="province inferred from recipient_org; not study-area geography",
)

topk_table = city_concentration.assign(
    section="top_k_city_share_by_institutions",
    field="recipient_org_inferred",
    rank=city_concentration["top_k"] if not city_concentration.empty else pd.Series(dtype=int),
    place_name=city_concentration["label"] if not city_concentration.empty else pd.Series(dtype=str),
    n_institutions=city_concentration["unique_orgs"] if not city_concentration.empty else pd.Series(dtype=int),
    note="cumulative share of institutions in top-k inferred recipient-organization cities",
)

source_table = source_counts.assign(
    section="location_source_distribution",
    field="recipient_org_inferred",
    place_name=source_counts["location_source"],
    n_institutions=source_counts["unique_orgs"],
    note="recipient_org inference rule distribution at institution level; no unresolved locations allowed",
)

review_table = review_counts.assign(
    section="geo_mapping_review_distribution",
    field="recipient_org_geo_mapping_review",
    place_name=review_counts["needs_review_label"],
    n_institutions=review_counts["unique_orgs"],
    note="needs_review flags from recipient_org_geo_mapping.csv at institution level",
)

english_mapping_table = english_mapping_counts.assign(
    section="english_name_mapping_distribution",
    field="recipient_org_english_map",
    place_name=english_mapping_counts["mapping_status"],
    n_institutions=english_mapping_counts["unique_orgs"],
    note="coverage of recipient_org_english_name_map.csv for the new main table",
)

base_table = institution_base.assign(
    section="institution_base",
    field="recipient_org_inferred",
    place_name=institution_base["institution_name"],
    city=institution_base["inferred_city"],
    province=institution_base["inferred_province"],
    n_institutions=1,
    note="one row per unique recipient institution; original recipient_org retained where English map is missing",
)

points_table = institution_points.assign(
    section="institution_points",
    field="recipient_org_inferred",
    place_name=institution_points["institution_name"],
    city=institution_points["inferred_city"],
    province=institution_points["inferred_province"],
    n_institutions=1,
    note="projected recipient-organization mapped coordinate plus deterministic jitter; not exact address or study area",
)

index_table = institution_index.assign(
    section="institution_index",
    field="recipient_org_inferred",
    place_name=institution_index["institution_name"],
    city=institution_index["inferred_city"],
    province=institution_index["inferred_province"],
    n_institutions=1,
    note="full institution audit index retained in CSV; not drawn as dense panel text for N=9222",
)

spatial_table = pd.concat(
    [coverage_table, city_table, province_table, topk_table, source_table, review_table, english_mapping_table, base_table, points_table, index_table],
    ignore_index=True,
    sort=False,
)

preferred_columns = [
    "section", "field", "institution_id", "rank", "place_name", "recipient_org", "recipient_org_en", "recipient_org_en_mapped", "recipient_org_name_source",
    "institution_name", "institution_name_en", "province", "city", "region", "inferred_city", "inferred_province", "lon", "lat",
    "city_projected_x", "city_projected_y", "city_web_mercator_x", "city_web_mercator_y", "jittered_x", "jittered_y", "jittered_web_mercator_x", "jittered_web_mercator_y", "jitter_hash",
    "jitter_order", "jitter_radius_m", "n_records", "share_records", "unique_orgs", "n_institutions",
    "share_institutions", "n_cities", "award_year_min", "award_year_max", "year_range", "award_ids",
    "location_source", "source_type", "confidence", "needs_review", "source_url", "review_note", "mapping_status", "match_term", "region_color_group", "region_sort_rank", "province_color_group", "province_sort_rank", "index_col", "index_row", "wrapped_institution_name",
    "index_line_count", "index_display", "note",
]
spatial_table = spatial_table[
    [col for col in preferred_columns if col in spatial_table.columns]
    + [col for col in spatial_table.columns if col not in preferred_columns]
]
spatial_table.to_csv(TABLE_PATH, index=False, encoding="utf-8-sig")

spatial_table.head(15)


,section,field,institution_id,rank,place_name,recipient_org,recipient_org_en,recipient_org_en_mapped,recipient_org_name_source,institution_name,...,geo_mapping_review_note,geo_mapping_record_count,geo_mapping_award_year_min,geo_mapping_award_year_max,geo_mapping_original_location_source,geo_mapping_last_checked_date,institution_sort_name,city_institution_count,jitter_max_radius_m,jitter_angle_rad
0,field_coverage,recipient_org,NaN,NaN,recipient organization,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,field_coverage,knowledge_prod_place,NaN,NaN,knowledge-production place,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,field_coverage,research_object_place,NaN,NaN,research-object place,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,field_coverage,beneficiary_city,NaN,NaN,beneficiary city,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,field_coverage,recipient_org_english_map,NaN,NaN,recipient organization English-name map,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,field_coverage,inferred_institution_city,NaN,NaN,inferred recipient-organization city,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,field_coverage,inferred_institution_province,NaN,NaN,inferred recipient-organization province,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,institution_city_distribution,recipient_org_inferred,NaN,1.0,Beijing,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,institution_city_distribution,recipient_org_inferred,NaN,2.0,Nanjing,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,institution_city_distribution,recipient_org_inferred,NaN,3.0,Shanghai,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Fig. 5 plotting helper logic has been merged into the final export cell below.
# Keep this placeholder cell to avoid stale Mapbox cache logic or old SVG raster fallback logic in the notebook.


In [8]:
# Draw Fig. 5.
# This cell redraws Fig. 5 directly from the latest spatial distribution table and overwrites only the four Fig. 5 figure formats.
# Panel a uses EPSG:3857 so the Mapbox Web Mercator raster, China boundaries, and institution points share one CRS.
from pathlib import Path
from io import BytesIO
import math
import os
import re
import textwrap
import xml.etree.ElementTree as ET
from urllib.parse import parse_qs, quote, urlparse

import numpy as np
import pandas as pd
import requests
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib import font_manager
import matplotlib.patheffects as path_effects
from pyproj import Transformer
from shapely.geometry import MultiLineString

try:
    import geopandas as gpd
except Exception as exc:
    raise RuntimeError('Fig5 requires geopandas to redraw the China basemap.') from exc

# Allow execution from the project root or code/ subdirectory.
CWD = Path.cwd()
ROOT = CWD if (CWD / 'output' / 'tables' / '06_spatial_distribution.csv').exists() else CWD.parent
TABLE_PATH = ROOT / 'output' / 'tables' / '06_spatial_distribution.csv'
CHINA_BASEMAP_PATH = ROOT / 'data' / 'china_province_basemap.geojson'
FIG_BASE = ROOT / 'output' / 'figures' / 'Fig5_spatial_distribution'

# Font and export settings.
for font_path in [
    Path('/Library/Fonts/Times New Roman.ttf'),
    Path('/System/Library/Fonts/Supplemental/Times New Roman.ttf'),
]:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont('Times New Roman', fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError('Times New Roman is required for Fig5 export.') from exc

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'font.size': 13.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
})

PALETTE = {
    'ink': '#161616',
    'muted': '#6B7178',
    'grid': '#DDE2E6',
    'light': '#E9EEF2',
    'blue': '#5B8DB8',
    'teal': '#5E9A8E',
    'gold': '#C8A45D',
    'orange': '#CC8150',
    'grey': '#8A8F94',
    'point': '#737373',
}
REGION_ORDER = [
    'Northeast China', 'North China', 'East China', 'Central China',
    'South China', 'Southwest China', 'Northwest China'
]
PROVINCE_REGION_MAP = {
    'Liaoning': 'Northeast China', 'Jilin': 'Northeast China', 'Heilongjiang': 'Northeast China',
    'Beijing': 'North China', 'Tianjin': 'North China', 'Hebei': 'North China', 'Shanxi': 'North China', 'Inner Mongolia': 'North China',
    'Shanghai': 'East China', 'Jiangsu': 'East China', 'Zhejiang': 'East China', 'Anhui': 'East China', 'Fujian': 'East China', 'Jiangxi': 'East China', 'Shandong': 'East China', 'Taiwan': 'East China',
    'Henan': 'Central China', 'Hubei': 'Central China', 'Hunan': 'Central China',
    'Guangdong': 'South China', 'Guangxi': 'South China', 'Hainan': 'South China', 'Hong Kong': 'South China', 'Macau': 'South China',
    'Chongqing': 'Southwest China', 'Sichuan': 'Southwest China', 'Guizhou': 'Southwest China', 'Yunnan': 'Southwest China', 'Tibet': 'Southwest China',
    'Shaanxi': 'Northwest China', 'Gansu': 'Northwest China', 'Qinghai': 'Northwest China', 'Ningxia': 'Northwest China', 'Xinjiang': 'Northwest China',
}
CHINA_PROVINCE_EN_BY_CN = {
    '北京市': 'Beijing', '天津市': 'Tianjin', '河北省': 'Hebei', '山西省': 'Shanxi', '内蒙古自治区': 'Inner Mongolia',
    '辽宁省': 'Liaoning', '吉林省': 'Jilin', '黑龙江省': 'Heilongjiang', '上海市': 'Shanghai', '江苏省': 'Jiangsu',
    '浙江省': 'Zhejiang', '安徽省': 'Anhui', '福建省': 'Fujian', '江西省': 'Jiangxi', '山东省': 'Shandong',
    '河南省': 'Henan', '湖北省': 'Hubei', '湖南省': 'Hunan', '广东省': 'Guangdong', '广西壮族自治区': 'Guangxi',
    '海南省': 'Hainan', '重庆市': 'Chongqing', '四川省': 'Sichuan', '贵州省': 'Guizhou', '云南省': 'Yunnan',
    '西藏自治区': 'Tibet', '陕西省': 'Shaanxi', '甘肃省': 'Gansu', '青海省': 'Qinghai', '宁夏回族自治区': 'Ningxia',
    '新疆维吾尔自治区': 'Xinjiang', '香港特别行政区': 'Hong Kong', '澳门特别行政区': 'Macau', '台湾省': 'Taiwan',
}
REGION_COLORS = {
    'Northeast China': '#8BA9C4',
    'North China': '#C5A260',
    'East China': '#5B9588',
    'Central China': '#8B75A4',
    'South China': '#C98255',
    'Southwest China': '#78A277',
    'Northwest China': '#B6858F',
}


def mix_color(color, target='#FFFFFF', amount=0.62):
    rgb = np.array(mpl.colors.to_rgb(color), dtype=float)
    target_rgb = np.array(mpl.colors.to_rgb(target), dtype=float)
    return mpl.colors.to_hex(rgb + (target_rgb - rgb) * amount).upper()


REGION_BASE_COLORS = REGION_COLORS.copy()
REGION_FILL_COLORS = {region: mix_color(REGION_COLORS[region], '#FFFFFF', 0.60) for region in REGION_ORDER}
REGION_TEXT_COLORS = {region: mix_color(REGION_COLORS[region], '#000000', 0.12) for region in REGION_ORDER}
BAR_COLORS = REGION_COLORS.copy()
MAP_POINT_FILL = PALETTE['point']
TOP10_CITY_RED = '#C9271E'
TOP10_CITY_FILL = '#FFF3EF'
MAPBOX_TOKEN_ENV_VAR = 'MAPBOX_ACCESS_TOKEN'
MAPBOX_WMTS_URL_ENV_VAR = 'FIG5_MAPBOX_WMTS_URL'
MAPBOX_TILE_TEMPLATE_ENV_VAR = 'FIG5_MAPBOX_TILE_URL_TEMPLATE'
MAPBOX_STYLE_OWNER_ENV_VAR = 'FIG5_MAPBOX_STYLE_OWNER'
MAPBOX_STYLE_ID_ENV_VAR = 'FIG5_MAPBOX_STYLE_ID'
MAPBOX_WMTS_URL_DEFAULT = 'https://api.mapbox.com/styles/v1/wang27623056/ckvezjv172ljc14ruua0veipv/wmts?access_token=pk.eyJ1Ijoid2FuZzI3NjIzMDU2IiwiYSI6ImNrcXJycnlqajBudHAybm14em8xZWpyYTAifQ.OOEJAKFE-SGxrr-Jq-qCqQ'
MAPBOX_TILE_ZOOM = 4
MAPBOX_BACKGROUND_ALPHA = 0.78
MAPBOX_CACHE_WRITE_ENABLED = False
WEB_MERCATOR_LIMIT = 20037508.342789244
LONLAT_TO_WEB_MERCATOR = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
DISALLOWED_PANEL_C_ABBREVIATIONS = {
    'HIT', 'BIT', 'HUST', 'SCUT', 'NUIST', 'SUSTech', 'CUG', 'CUGB', 'CAS', 'CEA', 'USTB', 'SIAT', 'IMHE'
}


def region_fill_color(region):
    return REGION_FILL_COLORS.get(str(region), '#EEF1F3')


def source_label(source):
    labels = {
        'recipient_org_geo_mapping_exact': 'Exact geo mapping table match',
        'recipient_org_text_place_cue': 'Text place cue fallback',
        'recipient_org_institution_lookup': 'Institution lookup fallback',
        'unresolved': 'Unresolved',
    }
    return labels.get(str(source), str(source).replace('_', ' ').title())


def scrub_mapbox_token(text):
    scrubbed = re.sub(r"(access_token=)[^&\s\"'>]+", r"\1[REDACTED]", str(text))
    return re.sub(r"pk\.[A-Za-z0-9._-]+", "[REDACTED]", scrubbed)


def _token_from_url(url):
    if not url:
        return ''
    try:
        values = parse_qs(urlparse(str(url)).query).get('access_token', [])
    except Exception:
        return ''
    return str(values[0]).strip() if values else ''


def get_runtime_mapbox_access_token():
    token = os.environ.get(MAPBOX_TOKEN_ENV_VAR, '').strip()
    if token:
        return token
    runtime_url = os.environ.get(MAPBOX_WMTS_URL_ENV_VAR, '').strip() or MAPBOX_WMTS_URL_DEFAULT
    return _token_from_url(runtime_url)


def _runtime_style_parts():
    owner = os.environ.get(MAPBOX_STYLE_OWNER_ENV_VAR, '').strip()
    style_id = os.environ.get(MAPBOX_STYLE_ID_ENV_VAR, '').strip()
    if owner and style_id:
        return quote(owner, safe=''), quote(style_id, safe='')
    return '', ''


def get_mapbox_wmts_url():
    """Return a runtime WMTS URL without persisting access tokens in the notebook or outputs."""
    runtime_url = os.environ.get(MAPBOX_WMTS_URL_ENV_VAR, '').strip()
    if runtime_url:
        return runtime_url
    if MAPBOX_WMTS_URL_DEFAULT:
        return MAPBOX_WMTS_URL_DEFAULT
    access_token = get_runtime_mapbox_access_token()
    owner, style_id = _runtime_style_parts()
    if access_token and owner and style_id:
        return f'https://api.mapbox.com/styles/v1/{owner}/{style_id}/wmts?access_token={access_token}'
    return None


def get_mapbox_direct_tile_template():
    runtime_template = os.environ.get(MAPBOX_TILE_TEMPLATE_ENV_VAR, '').strip()
    if runtime_template:
        return runtime_template
    access_token = get_runtime_mapbox_access_token()
    owner, style_id = _runtime_style_parts()
    if access_token and owner and style_id:
        return f'https://api.mapbox.com/styles/v1/{owner}/{style_id}/tiles/256/{{z}}/{{x}}/{{y}}?access_token={{access_token}}'
    return None


def get_mapbox_tile_template():
    runtime_template = os.environ.get(MAPBOX_TILE_TEMPLATE_ENV_VAR, '').strip()
    if runtime_template:
        return runtime_template, 'Mapbox XYZ tile background from runtime template'

    wmts_url = get_mapbox_wmts_url()
    if wmts_url:
        try:
            response = requests.get(wmts_url, timeout=45)
            response.raise_for_status()
            root = ET.fromstring(response.content)
            for elem in root.iter():
                if elem.tag.endswith('ResourceURL') and elem.attrib.get('resourceType') == 'tile':
                    template = elem.attrib.get('template')
                    if template:
                        return template, 'Mapbox WMTS background'
            return None, 'Mapbox WMTS background unavailable: no tile ResourceURL'
        except Exception as exc:
            direct_template = get_mapbox_direct_tile_template()
            if direct_template:
                return direct_template, f'Mapbox style tile background; WMTS parse failed: {scrub_mapbox_token(exc)}'
            return None, f'Mapbox WMTS background unavailable: {scrub_mapbox_token(exc)}'

    direct_template = get_mapbox_direct_tile_template()
    if direct_template:
        return direct_template, 'Mapbox style tile background from runtime style environment'
    return None, 'Mapbox WMTS background unavailable: set FIG5_MAPBOX_WMTS_URL or FIG5_MAPBOX_TILE_URL_TEMPLATE'


def materialize_tile_url(template, z, x, y):
    access_token = get_runtime_mapbox_access_token()
    replacements = {
        '{TileMatrixSet}': 'GoogleMapsCompatible',
        '{TileMatrix}': str(z),
        '{TileCol}': str(x),
        '{TileRow}': str(y),
        '{z}': str(z),
        '{x}': str(x),
        '{y}': str(y),
        '{access_token}': access_token,
    }
    url = str(template)
    for key, value in replacements.items():
        url = url.replace(key, value)
    if '{access_token}' in url and not access_token:
        raise RuntimeError(f'{MAPBOX_TOKEN_ENV_VAR} or a token-bearing {MAPBOX_WMTS_URL_ENV_VAR} is required for this tile template')
    return url


def tile_bounds_web_mercator(z, x, y):
    n = 2 ** z
    tile_span = 2 * WEB_MERCATOR_LIMIT / n
    xmin = -WEB_MERCATOR_LIMIT + x * tile_span
    xmax = xmin + tile_span
    ymax = WEB_MERCATOR_LIMIT - y * tile_span
    ymin = ymax - tile_span
    return xmin, ymin, xmax, ymax


def tile_range_for_web_mercator_bounds(xmin, ymin, xmax, ymax, z):
    n = 2 ** z

    def clamp(value):
        return max(0, min(n - 1, int(value)))

    x0 = clamp(math.floor((xmin + WEB_MERCATOR_LIMIT) / (2 * WEB_MERCATOR_LIMIT) * n))
    x1 = clamp(math.floor((xmax + WEB_MERCATOR_LIMIT) / (2 * WEB_MERCATOR_LIMIT) * n))
    y0 = clamp(math.floor((WEB_MERCATOR_LIMIT - ymax) / (2 * WEB_MERCATOR_LIMIT) * n))
    y1 = clamp(math.floor((WEB_MERCATOR_LIMIT - ymin) / (2 * WEB_MERCATOR_LIMIT) * n))
    return range(x0, x1 + 1), range(y0, y1 + 1)


def load_mapbox_basemap(web_mercator_bounds, zoom=MAPBOX_TILE_ZOOM):
    """Build a Mapbox tile mosaic whose extent is in EPSG:3857 meters."""
    template, status = get_mapbox_tile_template()
    if not template:
        return None, None, status
    xmin, ymin, xmax, ymax = [float(value) for value in web_mercator_bounds]
    x_range, y_range = tile_range_for_web_mercator_bounds(xmin, ymin, xmax, ymax, zoom)
    x_tiles = list(x_range)
    y_tiles = list(y_range)
    if not x_tiles or not y_tiles:
        return None, None, 'Mapbox WMTS background unavailable: no intersecting Web Mercator tiles'

    tile_images = {}
    first_size = None
    failures = []
    for tile_y in y_tiles:
        for tile_x in x_tiles:
            try:
                url = materialize_tile_url(template, zoom, tile_x, tile_y)
                response = requests.get(url, timeout=45)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content)).convert('RGB')
                if first_size is None:
                    first_size = image.size
                elif image.size != first_size:
                    image = image.resize(first_size, Image.Resampling.BILINEAR)
                tile_images[(tile_x, tile_y)] = image
            except Exception as exc:
                failures.append(f'z{zoom}/x{tile_x}/y{tile_y}: {scrub_mapbox_token(exc)}')
    if not tile_images:
        return None, None, f'Mapbox WMTS background unavailable: {failures[0] if failures else "no tiles loaded"}'
    if failures:
        return None, None, f'Mapbox WMTS background incomplete: {len(failures)} failed tile request(s); first failure: {failures[0]}'

    tile_w, tile_h = first_size
    mosaic = Image.new('RGB', (tile_w * len(x_tiles), tile_h * len(y_tiles)), 'white')
    for col, tile_x in enumerate(x_tiles):
        for row, tile_y in enumerate(y_tiles):
            image = tile_images.get((tile_x, tile_y))
            if image is not None:
                mosaic.paste(image, (col * tile_w, row * tile_h))
    extent = (
        tile_bounds_web_mercator(zoom, x_tiles[0], y_tiles[0])[0],
        tile_bounds_web_mercator(zoom, x_tiles[-1], y_tiles[-1])[2],
        tile_bounds_web_mercator(zoom, x_tiles[0], y_tiles[-1])[1],
        tile_bounds_web_mercator(zoom, x_tiles[0], y_tiles[0])[3],
    )
    status = f'{status}; CRS=EPSG:3857; zoom={zoom}; tiles={len(tile_images)}/{len(x_tiles) * len(y_tiles)}'
    if failures:
        status += f'; failed_tiles={len(failures)}'
    return np.asarray(mosaic), extent, status


PANEL_C_CSV_EXPECTED_ORGS = {
    '哈尔滨工程大学',
    '中国科学院长春应用化学研究所',
    '哈尔滨理工大学',
    '长春工业大学',
    '北京航空航天大学',
    '成都理工大学',
    '西安交通大学',
    '兰州大学',
    '西安建筑科技大学',
}

# Fallback only: formal data/recipient_org_english_name_map.csv names are always used first.
TOP_ORG_FULL_NAME_OVERRIDES = {
    '哈尔滨工程大学': 'Harbin Engineering University',
    '中国科学院长春应用化学研究所': 'Changchun Institute of Applied Chemistry, Chinese Academy of Sciences',
    '哈尔滨理工大学': 'Harbin University of Science and Technology',
    '长春工业大学': 'Changchun University of Technology',
    '哈尔滨师范大学': 'Harbin Normal University',
    '北京航空航天大学': 'Beihang University',
    '中国科学院南京地理与湖泊研究所': 'Nanjing Institute of Geography and Limnology, Chinese Academy of Sciences',
    '武汉科技大学': 'Wuhan University of Science and Technology',
    '桂林电子科技大学': 'Guilin University of Electronic Technology',
    '成都理工大学': 'Chengdu University of Technology',
    '西安交通大学': "Xi'an Jiaotong University",
    '兰州大学': 'Lanzhou University',
    '西安建筑科技大学': "Xi'an University of Architecture and Technology",
    '大连理工大学': 'Dalian University of Technology',
    '吉林大学': 'Jilin University',
    '东北大学': 'Northeastern University',
    '中国科学院东北地理与农业生态研究所': 'Northeast Institute of Geography and Agroecology, Chinese Academy of Sciences',
    '中国科学院空天信息创新研究院': 'Aerospace Information Research Institute, Chinese Academy of Sciences',
    '中国科学院大气物理研究所': 'Institute of Atmospheric Physics, Chinese Academy of Sciences',
    '复旦大学': 'Fudan University',
    '中国人民解放军国防科技大学': 'National University of Defense Technology',
    '暨南大学': 'Jinan University',
    '电子科技大学': 'University of Electronic Science and Technology of China',
    '西南大学': 'Southwest University',
    '中国科学院西北生态环境资源研究院': 'Northwest Institute of Eco-Environment and Resources, Chinese Academy of Sciences',
    '西北工业大学': 'Northwestern Polytechnical University',
    '新疆大学': 'Xinjiang University',
}


def _is_nonempty_name(value):
    text = '' if pd.isna(value) else str(value).strip()
    return text and text.lower() not in {'nan', 'none', 'null'}


def clean_full_institution_name(name):
    name = re.sub(r'\s+', ' ', str(name)).strip(' ,')
    name = name.replace('Chinese Acad. Sci.', 'Chinese Academy of Sciences')
    return name


def resolve_full_top_institution_name(row):
    for field in ['recipient_org_en', 'institution_name_en']:
        value = row.get(field, '')
        if _is_nonempty_name(value):
            name = clean_full_institution_name(value)
            if not re.search(r'[一-鿿]', name):
                return pd.Series({'org_full': name, 'org_full_source': 'recipient_org_english_name_map'})

    for field in ['recipient_org', 'institution_name', 'place_name']:
        value = row.get(field, '')
        if _is_nonempty_name(value):
            key = str(value).strip()
            if key in TOP_ORG_FULL_NAME_OVERRIDES:
                return pd.Series({'org_full': TOP_ORG_FULL_NAME_OVERRIDES[key], 'org_full_source': 'top_org_full_name_override_fallback'})

    for field in ['institution_display_name', 'place_name', 'institution_name', 'recipient_org']:
        value = row.get(field, '')
        if _is_nonempty_name(value):
            name = clean_full_institution_name(value)
            if not re.search(r'[一-鿿]', name):
                return pd.Series({'org_full': name, 'org_full_source': f'non_cjk_{field}'})
    raise ValueError(f"Panel c needs an English full-name mapping for institution_id={int(row['institution_id'])}")


def full_top_institution_name(row):
    return resolve_full_top_institution_name(row)['org_full']


def wrap_panel_c_name(name, width=32):
    lines = textwrap.wrap(str(name), width=width, break_long_words=False, break_on_hyphens=False)
    return '\n'.join(lines) if lines else str(name)


def build_region_top_institutions(index_df, top_n=8):
    table = index_df.dropna(subset=['region']).copy()
    table['n_records'] = pd.to_numeric(table['n_records'], errors='coerce').fillna(0).astype(int)
    pieces = []
    for region in REGION_ORDER:
        region_rows = (
            table.loc[table['region'].eq(region)]
            .sort_values(['n_records', 'institution_sort_name', 'institution_id'], ascending=[False, True, True], kind='mergesort')
            .head(top_n)
            .copy()
        )
        region_rows['regional_rank'] = np.arange(1, len(region_rows) + 1)
        resolved_names = region_rows.apply(resolve_full_top_institution_name, axis=1)
        region_rows = pd.concat([region_rows, resolved_names], axis=1)
        region_rows['org_label'] = region_rows['org_full'].map(wrap_panel_c_name)
        region_rows['org_label_lines'] = region_rows['org_label'].map(lambda value: max(1, str(value).count('\n') + 1))
        region_rows['org_short'] = region_rows['org_full']
        pieces.append(region_rows)
    top_table = pd.concat(pieces, ignore_index=True) if pieces else table.iloc[0:0].copy()
    cjk_rows = top_table.loc[top_table['org_full'].astype(str).str.contains(r'[一-鿿]', regex=True, na=False)]
    if not cjk_rows.empty:
        raise ValueError(f"Panel c full-name mapping still contains CJK text: {cjk_rows[['region', 'institution_name', 'org_full']].to_dict('records')}")
    abbrev_pattern = r'\b(?:' + '|'.join(re.escape(term) for term in sorted(DISALLOWED_PANEL_C_ABBREVIATIONS, key=len, reverse=True)) + r')\b'
    abbrev_rows = top_table.loc[top_table['org_full'].astype(str).str.contains(abbrev_pattern, regex=True, na=False)]
    if not abbrev_rows.empty:
        raise ValueError(f"Panel c full-name mapping still contains abbreviations: {abbrev_rows[['region', 'org_full']].to_dict('records')}")
    return top_table


def contrast_text_color(fill_color):
    r, g, b = mpl.colors.to_rgb(fill_color)
    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return '#111111' if luminance > 0.58 else 'white'


def web_mercator_y_to_latitude(y_value):
    radius_m = WEB_MERCATOR_LIMIT / math.pi
    return math.degrees(2 * math.atan(math.exp(float(y_value) / radius_m)) - math.pi / 2)


def draw_map_scale_bar(ax, map_xlim, map_ylim, length_km=1500):
    x_span = map_xlim[1] - map_xlim[0]
    y_span = map_ylim[1] - map_ylim[0]
    x0 = map_xlim[0] + 0.070 * x_span
    y0 = map_ylim[0] + 0.060 * y_span
    local_lat = web_mercator_y_to_latitude(y0)
    projected_length = length_km * 1000 / max(math.cos(math.radians(local_lat)), 0.25)
    tick_h = 0.013 * y_span
    ax.plot([x0, x0 + projected_length], [y0, y0], color='#111111', linewidth=1.75, solid_capstyle='butt', zorder=12)
    ax.plot([x0, x0], [y0 - tick_h, y0 + tick_h], color='#111111', linewidth=1.35, zorder=12)
    ax.plot([x0 + projected_length, x0 + projected_length], [y0 - tick_h, y0 + tick_h], color='#111111', linewidth=1.35, zorder=12)
    label_y = y0 - 0.030 * y_span
    text_effect = [path_effects.withStroke(linewidth=2.8, foreground='white')]
    ax.text(x0, label_y, '0', ha='center', va='top', fontsize=12.0, color='#111111', path_effects=text_effect, zorder=13)
    ax.text(x0 + projected_length, label_y, f'{length_km:,} km', ha='center', va='top', fontsize=12.0, color='#111111', path_effects=text_effect, zorder=13)


def draw_top_city_map_annotations(ax, city_table):
    top10 = city_table.sort_values('rank').head(10).copy()
    for col in ['city_web_mercator_x', 'city_web_mercator_y', 'unique_orgs']:
        top10[col] = pd.to_numeric(top10[col], errors='coerce')
    top10 = top10.dropna(subset=['city_web_mercator_x', 'city_web_mercator_y', 'unique_orgs']).copy()
    if top10.empty:
        return
    max_orgs = max(float(top10['unique_orgs'].max()), 1.0)
    sizes = 58 + 148 * np.sqrt(top10['unique_orgs'].astype(float) / max_orgs)
    ax.scatter(
        top10['city_web_mercator_x'], top10['city_web_mercator_y'],
        s=sizes, facecolors=TOP10_CITY_FILL, edgecolors=TOP10_CITY_RED,
        linewidths=1.30, alpha=0.96, zorder=9,
    )
    label_offsets = {
        'Beijing': (-24, 18), 'Shanghai': (24, 8), 'Nanjing': (-24, 16),
        'Guangzhou': (-24, 18), 'Tianjin': (20, -8), "Xi'an": (-28, 16),
        'Shenzhen': (22, -18), 'Wuhan': (-30, 8), 'Shenyang': (22, 14),
        'Chongqing': (-32, -12),
    }
    text_effect = [path_effects.withStroke(linewidth=2.9, foreground='white')]
    for _, rec in top10.iterrows():
        city = str(rec['place_name'])
        dx, dy = label_offsets.get(city, (28, 16))
        ax.annotate(
            city,
            xy=(float(rec['city_web_mercator_x']), float(rec['city_web_mercator_y'])),
            xytext=(dx, dy), textcoords='offset points',
            ha='left' if dx >= 0 else 'right', va='center',
            fontsize=11.4, fontweight='bold', color=TOP10_CITY_RED,
            arrowprops={'arrowstyle': '-', 'color': TOP10_CITY_RED, 'linewidth': 0.92, 'shrinkA': 0, 'shrinkB': 7},
            path_effects=text_effect, zorder=13,
        )


def validate_institution_point_coordinates(points_df):
    required = ['lon', 'lat', 'city_web_mercator_x', 'city_web_mercator_y', 'jittered_web_mercator_x', 'jittered_web_mercator_y', 'jitter_radius_m']
    missing = [col for col in required if col not in points_df.columns]
    if missing:
        raise ValueError(f'Panel a coordinate validation missing columns: {missing}')
    coord = points_df[required].copy()
    for col in required:
        coord[col] = pd.to_numeric(coord[col], errors='coerce')
    if coord[['lon', 'lat', 'city_web_mercator_x', 'city_web_mercator_y']].isna().any().any():
        raise ValueError('Panel a coordinate validation found missing lon/lat or Web Mercator coordinates.')
    if not coord['lon'].between(73, 136).all() or not coord['lat'].between(18, 54).all():
        bad = coord.loc[~coord['lon'].between(73, 136) | ~coord['lat'].between(18, 54)].head(5).to_dict('records')
        raise ValueError(f'Panel a coordinate validation found out-of-China lon/lat values: {bad}')
    expected_x, expected_y = LONLAT_TO_WEB_MERCATOR.transform(coord['lon'].to_numpy(), coord['lat'].to_numpy())
    max_x_error = float(np.nanmax(np.abs(expected_x - coord['city_web_mercator_x'].to_numpy())))
    max_y_error = float(np.nanmax(np.abs(expected_y - coord['city_web_mercator_y'].to_numpy())))
    if max_x_error > 1e-4 or max_y_error > 1e-4:
        raise ValueError(f'Panel a coordinate validation failed Web Mercator reprojection check: dx={max_x_error}, dy={max_y_error}')
    jitter_distance = np.hypot(
        coord['jittered_web_mercator_x'].to_numpy() - coord['city_web_mercator_x'].to_numpy(),
        coord['jittered_web_mercator_y'].to_numpy() - coord['city_web_mercator_y'].to_numpy(),
    )
    jitter_error = float(np.nanmax(np.abs(jitter_distance - coord['jitter_radius_m'].to_numpy())))
    if jitter_error > 1e-4:
        raise ValueError(f'Panel a coordinate validation failed jitter-radius check: max error={jitter_error}')
    return {
        'n_points': int(len(coord)),
        'lon_range': (float(coord['lon'].min()), float(coord['lon'].max())),
        'lat_range': (float(coord['lat'].min()), float(coord['lat'].max())),
        'max_web_mercator_error_m': max(max_x_error, max_y_error),
        'max_jitter_error_m': jitter_error,
    }


def normalize_south_china_sea_nine_dash(china_gdf):
    fixed = china_gdf.copy()
    fixed['boundary_role'] = ''
    check = {
        'candidate_rows': 0,
        'original_dash_count': None,
        'plotted_dash_count': None,
        'removed_dash_bounds_lonlat': None,
    }
    boundary_rows = fixed.loc[fixed['name'].astype(str).eq('境界线')]
    for idx, geom in boundary_rows.geometry.items():
        if geom is None or geom.is_empty or geom.geom_type != 'MultiLineString':
            continue
        minx, miny, maxx, maxy = geom.bounds
        lines = list(geom.geoms)
        is_south_china_sea = minx < 109 and miny < 4 and maxx > 122 and maxy > 24 and len(lines) >= 9
        if not is_south_china_sea:
            continue
        check['candidate_rows'] += 1
        check['original_dash_count'] = len(lines)
        if len(lines) == 10:
            extra_idx = max(range(len(lines)), key=lambda pos: lines[pos].bounds[3])
            check['removed_dash_bounds_lonlat'] = tuple(round(float(v), 6) for v in lines[extra_idx].bounds)
            lines = [line for pos, line in enumerate(lines) if pos != extra_idx]
        if len(lines) != 9:
            raise ValueError(f'South China Sea boundary should render as nine dashes; got {len(lines)} from source row {idx}.')
        fixed.at[idx, 'geometry'] = MultiLineString(lines)
        fixed.at[idx, 'boundary_role'] = 'south_china_sea_nine_dash'
        check['plotted_dash_count'] = len(lines)
    if check['candidate_rows'] != 1:
        raise ValueError(f'Expected exactly one South China Sea boundary candidate; got {check["candidate_rows"]}.')
    return fixed, check


def draw_region_top_institutions_panel(ax, top_table):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.000, 1.012, 'c', transform=ax.transAxes, ha='left', va='bottom', fontsize=18.4, fontweight='bold', color=PALETTE['ink'])
    ax.text(0.032, 1.012, 'Top 8 recipient organizations by province region', transform=ax.transAxes, ha='left', va='bottom', fontsize=18.4, fontweight='bold', color=PALETTE['ink'])

    left_pad = 0.004
    col_gap = 0.018
    block_w = 0.226
    right_pad = 1 - left_pad - 3 * col_gap - 4 * block_w
    if right_pad < 0:
        raise ValueError('Panel c block layout exceeds the available width.')
    row_gap = 0.018
    top_y = 0.988
    block_h = 0.500
    header_h = 0.045
    content_top_pad = 0.008
    content_bottom_pad = 0.012
    content_fill_frac = 0.90
    visual_block_h = header_h + content_top_pad + (block_h - header_h - content_top_pad - content_bottom_pad) * content_fill_frac + 0.006
    row_top_by_row = {0: top_y, 1: top_y - visual_block_h - row_gap}
    name_pad = 0.008
    count_x_frac = 0.955
    slot_by_region = {
        'Northeast China': (0, 0),
        'North China': (0, 1),
        'East China': (0, 2),
        'Central China': (0, 3),
        'South China': (1, 0),
        'Southwest China': (1, 1),
        'Northwest China': (1, 2),
    }

    def slot_xy(slot):
        row, col = slot
        x0 = left_pad + col * (block_w + col_gap)
        y0 = row_top_by_row[row] - block_h
        return x0, y0

    for region in REGION_ORDER:
        x0, y0 = slot_xy(slot_by_region[region])
        block_top = y0 + block_h
        header_y0 = block_top - header_h
        base = REGION_COLORS[region]
        header_color = '#000000'
        count_x = x0 + block_w * count_x_frac

        region_rows = top_table.loc[top_table['region'].eq(region)].sort_values('regional_rank')
        inner_top = header_y0 - content_top_pad
        inner_bottom = y0 + content_bottom_pad
        available_h = max(inner_top - inner_bottom, 0.01)
        row_units = (region_rows['org_label_lines'].astype(float) * 1.00 + 0.30).tolist()
        unit_h = (available_h * content_fill_frac) / max(sum(row_units), 1)
        cursor = inner_top
        row_specs = []
        for (_, rec), units in zip(region_rows.iterrows(), row_units):
            row_h = units * unit_h
            yy = cursor - row_h / 2
            rank = int(rec['regional_rank'])
            sep_y = cursor + 0.0015 if rank > 1 else None
            label = f"{rank}. {rec['org_label']}"
            row_specs.append((label, yy, sep_y, int(rec['n_records'])))
            cursor -= row_h

        visual_bottom = max(y0, cursor - 0.006)
        ax.add_patch(Rectangle((x0, visual_bottom), block_w, block_top - visual_bottom, transform=ax.transAxes, facecolor='#FBFCFD', edgecolor='#D7DDE2', linewidth=0.42, zorder=0))
        ax.add_patch(Rectangle((x0, header_y0), block_w, header_h, transform=ax.transAxes, facecolor=base, edgecolor=base, linewidth=0.0, zorder=1))
        ax.text(x0 + name_pad, header_y0 + header_h / 2, region, transform=ax.transAxes, ha='left', va='center', fontsize=13.2, fontweight='bold', color=header_color)
        ax.text(count_x, header_y0 + header_h / 2, 'Count', transform=ax.transAxes, ha='right', va='center', fontsize=12.3, color=header_color)

        for label, yy, sep_y, count_value in row_specs:
            if sep_y is not None:
                ax.plot([x0 + 0.007, x0 + block_w - 0.007], [sep_y, sep_y], transform=ax.transAxes, color='#E5E9EC', linewidth=0.30, zorder=1)
            ax.text(x0 + name_pad, yy, label, transform=ax.transAxes, ha='left', va='center', fontsize=10.8, linespacing=0.82, color='#111111', clip_on=True)
            ax.text(count_x, yy, f"{count_value:,}", transform=ax.transAxes, ha='right', va='center', fontsize=10.8, fontweight='bold', color='#111111', clip_on=True)


spatial = pd.read_csv(TABLE_PATH)
points = spatial.loc[spatial['section'].eq('institution_points')].copy()
city_top = spatial.loc[spatial['section'].eq('institution_city_distribution')].sort_values('rank').head(20).copy()
coverage = spatial.loc[spatial['section'].eq('field_coverage')].copy()
source = spatial.loc[spatial['section'].eq('location_source_distribution')].copy()
english = spatial.loc[spatial['section'].eq('english_name_mapping_distribution')].copy()
institution_index = spatial.loc[spatial['section'].eq('institution_index')].copy()

for col in ['lon', 'lat', 'city_web_mercator_x', 'city_web_mercator_y', 'jittered_web_mercator_x', 'jittered_web_mercator_y', 'jitter_radius_m', 'n_records', 'unique_orgs']:
    if col in points.columns:
        points[col] = pd.to_numeric(points[col], errors='coerce')
points = points.dropna(subset=['jittered_web_mercator_x', 'jittered_web_mercator_y']).copy()
if points.empty:
    raise ValueError('Panel a requires institution points with EPSG:3857 coordinates.')
panel_a_coordinate_check = validate_institution_point_coordinates(points)

n_records = int(coverage.loc[coverage['field'].eq('recipient_org'), 'n_records'].iloc[0])
institution_base = spatial.loc[spatial['section'].eq('institution_base')].copy()
n_institutions = int(institution_base['institution_id'].nunique())
located_unique = int(points['institution_id'].nunique())
inferred_records = int(coverage.loc[coverage['field'].eq('inferred_institution_city'), 'n_records'].iloc[0])
english_records = int(coverage.loc[coverage['field'].eq('recipient_org_english_map'), 'n_records'].iloc[0])
structured_records = int(coverage.loc[coverage['field'].isin(['knowledge_prod_place', 'research_object_place', 'beneficiary_city']), 'n_records'].sum())
english_unique = int(english.loc[english['place_name'].astype(str).str.contains('mapped English name', na=False), 'unique_orgs'].sum())
region_top10 = build_region_top_institutions(institution_index, top_n=8)
region_top10_counts = region_top10.groupby('region').size().reindex(REGION_ORDER, fill_value=0)
if not region_top10_counts.eq(8).all():
    raise ValueError(f"Panel c requires exactly 8 organizations per region: {region_top10_counts.to_dict()}")

china_raw = gpd.read_file(CHINA_BASEMAP_PATH)
if china_raw.crs is None:
    china_raw = china_raw.set_crs(epsg=4490)
china_raw, south_china_sea_nine_dash_check = normalize_south_china_sea_nine_dash(china_raw)
china = china_raw.to_crs(epsg=3857).copy()
if china.crs is None or china.crs.to_epsg() != 3857:
    raise ValueError(f'China basemap must be EPSG:3857 for Fig5a; got {china.crs}')
china['province_en'] = china['name'].map(CHINA_PROVINCE_EN_BY_CN)
china['region'] = china['province_en'].map(PROVINCE_REGION_MAP)
china['fill_color'] = china['region'].map(REGION_FILL_COLORS).fillna('#EEF1F3')
polygons = china.loc[~china['name'].astype(str).eq('境界线')].copy()
boundary = china.loc[china['name'].astype(str).eq('境界线')].copy()
south_china_sea_boundary = boundary.loc[boundary['boundary_role'].eq('south_china_sea_nine_dash')].copy()
other_boundary = boundary.loc[~boundary['boundary_role'].eq('south_china_sea_nine_dash')].copy()

fig = plt.figure(figsize=(14.60, 18.80), facecolor='white')
# Manual axes: panels a and b share the first row; panel c spans the lower row as a compact 2 x 4 regional grid.
left_x = 0.045
map_x = left_x
map_w = 0.590
city_x = 0.700
city_w = 0.250
top_row_y = 0.570
top_row_h = 0.380
panel_c_x = 0.060
panel_c_y = 0.050
panel_c_w = 0.910
panel_c_h = 0.470
ax_map = fig.add_axes([map_x, top_row_y, map_w, top_row_h])
ax_city = fig.add_axes([city_x, top_row_y, city_w, top_row_h])
ax_top_orgs = fig.add_axes([panel_c_x, panel_c_y, panel_c_w, panel_c_h])

# Panel a: all layers are Web Mercator. The raster extent is the EPSG:3857 tile mosaic extent.
bounds = china.total_bounds
width = bounds[2] - bounds[0]
height = bounds[3] - bounds[1]
map_xlim = (bounds[0] - 0.018 * width, bounds[2] + 0.045 * width)
map_ylim = (bounds[1] - 0.018 * height, bounds[3] + 0.030 * height)
mapbox_bounds = (map_xlim[0], map_ylim[0], map_xlim[1], map_ylim[1])
mapbox_image, mapbox_extent, mapbox_status = load_mapbox_basemap(mapbox_bounds)
if mapbox_image is None:
    raise RuntimeError(f'Fig5a Mapbox WMTS basemap failed; export stopped before fallback rendering: {mapbox_status}')
mapbox_background_available = mapbox_image is not None
if mapbox_background_available:
    ax_map.imshow(mapbox_image, extent=mapbox_extent, origin='upper', zorder=0, alpha=MAPBOX_BACKGROUND_ALPHA)
    province_alpha = 0.50
else:
    ax_map.set_facecolor('#F7F8F9')
    province_alpha = 0.72
polygons.plot(ax=ax_map, color=polygons['fill_color'].tolist(), edgecolor='#FFFFFF', linewidth=0.28, alpha=province_alpha, zorder=1)
polygons.boundary.plot(ax=ax_map, color='#9EA6AE', linewidth=0.24, alpha=0.88, zorder=2)
if not other_boundary.empty:
    other_boundary.plot(ax=ax_map, color='#3E454B', linewidth=0.42, alpha=0.78, linestyle=(0, (4, 3)), zorder=3)
if not south_china_sea_boundary.empty:
    south_china_sea_boundary.plot(ax=ax_map, color='#3E454B', linewidth=0.42, alpha=0.78, linestyle='solid', zorder=3)
ax_map.scatter(points['jittered_web_mercator_x'], points['jittered_web_mercator_y'], s=18.0, color=MAP_POINT_FILL, alpha=0.64, edgecolors='white', linewidths=0.30, zorder=5)
ax_map.set_xlim(*map_xlim)
ax_map.set_ylim(*map_ylim)
ax_map.set_aspect('equal', adjustable='box')
ax_map.axis('off')
ax_map.text(-0.025, 1.018, 'a', transform=ax_map.transAxes, ha='left', va='bottom', fontsize=18.0, fontweight='bold', color=PALETTE['ink'])
ax_map.text(0.020, 1.018, 'Recipient-organization locations', transform=ax_map.transAxes, ha='left', va='bottom', fontsize=18.0, fontweight='bold', color=PALETTE['ink'])
legend_handles = [Patch(facecolor=REGION_COLORS[r], edgecolor='none', label=r) for r in REGION_ORDER]
legend = ax_map.legend(handles=legend_handles, title='Province region', loc='lower left', bbox_to_anchor=(0.070, 0.122), bbox_transform=ax_map.transAxes, fontsize=11.6, title_fontsize=12.3, handlelength=1.38, handleheight=1.38, labelspacing=0.42, borderpad=0.0)
legend._legend_box.align = 'left'
legend.get_title().set_fontweight('bold')
legend.get_title().set_ha('left')
legend.get_title().set_multialignment('left')
legend.set_zorder(10)
draw_top_city_map_annotations(ax_map, city_top)
draw_map_scale_bar(ax_map, map_xlim, map_ylim, length_km=1500)

# Panel b: top-city bars; bottom explanatory small note intentionally removed.
city_bar = city_top.iloc[::-1].copy()
city_bar['rank'] = pd.to_numeric(city_bar['rank'], errors='coerce')
top10_city_mask = city_bar['rank'].le(10)
y = np.arange(len(city_bar))
bar_colors = [BAR_COLORS.get(str(region), PALETTE['grey']) for region in city_bar['region']]
ax_city.barh(y, city_bar['unique_orgs'].astype(float).values, color=bar_colors, alpha=0.94, height=0.62)
ax_city.set_yticks(y)
ax_city.set_yticklabels(city_bar['place_name'].astype(str).values, fontsize=12.0)
ax_city.grid(axis='x', color=PALETTE['grid'], linewidth=0.6)
ax_city.set_axisbelow(True)
ax_city.tick_params(axis='both', length=0, labelsize=11.6, colors='#000000')
for tick_label, is_top10 in zip(ax_city.get_yticklabels(), top10_city_mask.tolist()):
    tick_label.set_color(TOP10_CITY_RED if is_top10 else '#000000')
    tick_label.set_fontweight('normal')
ax_city.spines['left'].set_color('#000000')
ax_city.spines['bottom'].set_color('#000000')
for yy, value, is_top10 in zip(y, city_bar['unique_orgs'].astype(int).values, top10_city_mask.tolist()):
    ax_city.text(value + 0.7, yy, str(value), va='center', ha='left', fontsize=11.4, fontweight='normal', color='#000000')
ax_city.set_xlim(0, max(float(city_top['unique_orgs'].max()) * 1.23, 5))
ax_city.set_ylim(-0.65, len(city_bar) - 0.35)
ax_city.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
ax_city.set_xlabel('Unique recipient organizations', fontsize=13.8, labelpad=10, color='#000000')
ax_city.text(-0.025, 1.018, 'b', transform=ax_city.transAxes, ha='left', va='bottom', fontsize=18.0, fontweight='bold', color=PALETTE['ink'])
ax_city.text(0.020, 1.018, 'Top inferred cities', transform=ax_city.transAxes, ha='left', va='bottom', fontsize=18.0, fontweight='bold', color=PALETTE['ink'])

# Panel c: regional top recipient organizations with English full names.
draw_region_top_institutions_panel(ax_top_orgs, region_top10)

fig.savefig(FIG_BASE.with_suffix('.svg'), bbox_inches='tight', pad_inches=0.01, facecolor='white')
fig.savefig(FIG_BASE.with_suffix('.pdf'), bbox_inches='tight', pad_inches=0.01, facecolor='white')
fig.savefig(FIG_BASE.with_suffix('.png'), dpi=600, bbox_inches='tight', pad_inches=0.01, facecolor='white')
fig.savefig(FIG_BASE.with_suffix('.tiff'), dpi=600, bbox_inches='tight', pad_inches=0.01, facecolor='white', pil_kwargs={'compression': 'tiff_lzw'})
plt.close(fig)

figure_mode = 'mapbox_wmts_background' if mapbox_background_available else 'vector_map_no_mapbox_background'
mapbox_background_status = mapbox_status
figure_mode


'mapbox_wmts_background'

In [9]:
# Write the draft results paragraph and method notes.
# The log uses English for direct reuse in the review manuscript and states that points are jittered recipient-organization city centers, not study areas or exact addresses.
def fmt_pct(value):
    return f"{value * 100:.1f}%"

structured_coverage = coverage_df.loc[coverage_df["field"].isin(STRUCTURED_PLACE_FIELDS)].copy()
city_cov = coverage_df.loc[coverage_df["field"].eq("inferred_institution_city"), "coverage_pct"].iloc[0] / 100
institution_city_cov = float(institution_base["inferred_city"].notna().mean())
english_record_cov = coverage_df.loc[coverage_df["field"].eq("recipient_org_english_map"), "coverage_pct"].iloc[0] / 100
english_institution_cov = n_mapped_english_institutions / n_institutions if n_institutions else 0.0

if "region_top10" not in globals():
    region_top10 = build_region_top_institutions(institution_index, top_n=8)

top_city_parts = [
    f"{row.inferred_city} ({int(row.unique_orgs)} institutions; {int(row.n_records)} records)"
    for row in city_counts.head(5).itertuples()
]
concentration_lookup = city_concentration.set_index("top_k") if not city_concentration.empty else pd.DataFrame()
top5_inst_share = float(concentration_lookup.loc[5, "share_institutions"]) if 5 in concentration_lookup.index else 0.0
top10_inst_share = float(concentration_lookup.loc[10, "share_institutions"]) if 10 in concentration_lookup.index else 0.0
top20_inst_share = float(concentration_lookup.loc[20, "share_institutions"]) if 20 in concentration_lookup.index else 0.0

all_structured_empty = bool((structured_coverage["n_nonempty"] == 0).all())
structured_place_text = (
    "knowledge-production place, research-object place, and beneficiary-city fields each had 0 non-empty records"
    if all_structured_empty
    else "structured place fields were incomplete and are reported in the coverage table"
)
if figure_mode in {"mapbox_map", "mapbox_wmts_background"}:
    figure_mode_text = "China province basemap and recipient-organization points reprojected to EPSG:3857 over a Web Mercator Mapbox WMTS background"
elif figure_mode == "vector_map_no_mapbox_background":
    figure_mode_text = "China province basemap and recipient-organization points reprojected to EPSG:3857; Mapbox raster background unavailable at runtime"
else:
    figure_mode_text = "fallback recipient-organization-location scatter without a basemap"

top_city_sentence = (
    f"By unique institutions, the leading inferred cities were {', '.join(top_city_parts[:3])}"
    + (f", followed by {', '.join(top_city_parts[3:5])}" if len(top_city_parts) > 3 else "")
    + "."
    if top_city_parts
    else "No city ranking was produced because no recipient-organization cities were inferred."
)

region_leaders = region_top10.loc[region_top10["regional_rank"].eq(1)].copy()
region_leader_sentence = "; ".join(
    f"{row.region}: {row.org_short} ({int(row.n_records)} records)"
    for row in region_leaders.itertuples()
)

results_paragraph = (
    f"The updated Fig. 5 uses the new deduplicated main corpus (N = {n_records}) and treats recipient organization names as the reproducible spatial cue for institutional geography. "
    f"Structured study-place metadata remain unavailable for spatial mapping: {structured_place_text}. "
    f"The institution-level dataset contains {n_institutions} unique recipient organizations. "
    "Institutional locations were harmonized to city and province attributes for aggregate recipient-organization mapping. "
    f"{top_city_sentence} The top 5, top 10, and top 20 inferred cities contained "
    f"{fmt_pct(top5_inst_share)}, {fmt_pct(top10_inst_share)}, and {fmt_pct(top20_inst_share)} of all recipient institutions, respectively. "
    f"Panel c summarizes the top 8 recipient organizations by record count within each inferred province region; the leading regional recipients are {region_leader_sentence}. "
    f"These results describe where funded recipient institutions are located rather than verified study areas, knowledge-production places, or beneficiary cities."
)

caption = (
    "Fig. 5. Recipient-organization geography in the updated deduplicated corpus. "
    "Panel a maps each unique recipient institution with an inferred city as a fixed-size point jittered around the mapped WGS84 coordinate; points are not exact addresses and do not represent study areas. "
    "Panel b ranks the top 15 inferred cities by unique recipient institutions. "
    "Panel c lists the eight recipient organizations with the highest record counts within each inferred province region in seven compact right-side region blocks; the full institution index is retained in the source data table."
)

coverage_md = coverage_df.copy()
coverage_md = coverage_md[["field_label", "n_nonempty"]]

city_md = city_counts.head(15).copy()
city_md["share_institutions"] = city_md["share_institutions"].map(lambda x: f"{x * 100:.1f}%")
city_md["share_records"] = city_md["share_records"].map(lambda x: f"{x * 100:.1f}%")
city_md = city_md[["rank", "inferred_city", "inferred_province", "unique_orgs", "share_institutions", "n_records", "share_records"]]

topk_md = city_concentration.copy()
if not topk_md.empty:
    topk_md["share_institutions"] = topk_md["share_institutions"].map(lambda x: f"{x * 100:.1f}%")
    topk_md["share_records"] = topk_md["share_records"].map(lambda x: f"{x * 100:.1f}%")
    topk_md = topk_md[["top_k", "label", "unique_orgs", "share_institutions", "n_records", "share_records"]]

source_md = source_counts.copy()
source_md["location_source"] = source_md["location_source"].map(source_label)
source_md = source_md[["rank", "location_source", "unique_orgs", "n_records"]]

review_md = review_counts.copy()
review_md["share_institutions"] = review_md["share_institutions"].map(lambda x: f"{x * 100:.1f}%")
review_md["share_records"] = review_md["share_records"].map(lambda x: f"{x * 100:.1f}%")
review_md = review_md[["rank", "needs_review_label", "unique_orgs", "share_institutions", "n_records", "share_records"]]

english_md = english_mapping_counts.copy()
english_md["share_institutions"] = english_md["share_institutions"].map(lambda x: f"{x * 100:.1f}%")
english_md["share_records"] = english_md["share_records"].map(lambda x: f"{x * 100:.1f}%")
english_md = english_md[["rank", "mapping_status", "unique_orgs", "share_institutions", "n_records", "share_records"]]

region_top10_md = region_top10.copy()
region_top10_md = region_top10_md[["region", "regional_rank", "org_short", "org_full_source", "n_records"]].rename(columns={
    "regional_rank": "rank",
    "org_short": "organization",
    "org_full_source": "name_source",
    "n_records": "records",
})

max_jitter_km = float(institution_points["jitter_radius_m"].max() / 1000) if not institution_points.empty else 0.0
mapbox_status_text = scrub_mapbox_token(mapbox_background_status)
region_palette_md = pd.DataFrame([
    {
        "region": region,
        "fig7_base_color": REGION_BASE_COLORS[region],
        "map_fill_color": REGION_FILL_COLORS[region],
        "index_text_color": REGION_TEXT_COLORS[region],
    }
    for region in REGION_ORDER
])

log_text = f"""# 06 Spatial Distribution Results Draft

## Figure inputs and method notes

- Source file: project main corpus CSV.
- Total deduplicated records: {n_records}
- Unique recipient institutions: {n_institutions}
- English-name mapped institutions: {n_mapped_english_institutions}
- English-name unmapped institutions retained as source recipient-organization names: {n_unmapped_english_institutions}
- Structured place fields for knowledge production, research object and beneficiary city contain no non-empty values in the current analytical dataset.
- Figure mode used: {figure_mode_text}
- Region palette: seven low-saturation region colors derived from the Fig. 7 Sankey palette
- Mapbox WMTS background: {mapbox_status_text}
- Mapbox raster CRS: Web Mercator EPSG:3857; vector layers and points use the same projected CRS
- Mapbox cache write enabled: {MAPBOX_CACHE_WRITE_ENABLED}
- Deterministic jitter: stable SHA-256 hash; radius capped at 50 km around mapped recipient-organization coordinates
- Figure files: `output/figures/Fig5_spatial_distribution.svg/.pdf/.tiff/.png`
- Spatial distribution table: `output/tables/06_spatial_distribution.csv`

## Spatial Fields Used for Interpretation

{coverage_md.to_markdown(index=False)}

## Top Cities by Unique Recipient Institutions

{city_md.to_markdown(index=False)}

## Top-K City Share by Institutions

{topk_md.to_markdown(index=False) if not topk_md.empty else "No top-k city concentration table was produced."}

## Top 8 Recipient Organizations by Province Region

{region_top10_md.to_markdown(index=False)}

## Location Inference Source

{source_md.to_markdown(index=False)}

## English-Name Mapping Coverage

{english_md.to_markdown(index=False)}

## Region Palette

{region_palette_md.to_markdown(index=False)}

## Results Paragraph Draft

{results_paragraph}

## Figure Caption Draft

{caption}

## Interpretation Notes

- The three structured place fields are retained in the new main table but are empty in the current data, so Fig. 5 should be interpreted as recipient-organization geography, not as verified study-area, knowledge-production-place, or beneficiary-city geography.
- Mapped coordinates are retained from recipient_org_geo_mapping.csv and may represent city centroids or organization-level public coordinates depending on source_type. Panel a uses deterministic jitter only to separate overlapping institutions visually; it does not show exact addresses.
- Recipient-organization inference first uses exact `recipient_org_geo_mapping.csv` matches; CITY_META rules are retained only as a controlled fallback for future data updates.
- City bars rank unique recipient institutions. Record counts remain available as auxiliary fields in the table and log.
- Panel a uses the configured Mapbox WMTS/XYZ background in EPSG:3857. If the raster background is unavailable, the notebook stops instead of exporting a vector-only fallback; no token is persisted in SVG text, logs, CSV outputs, or cache files.
- The full institution-level index is kept in `output/tables/06_spatial_distribution.csv`; panel c uses seven compact right-side regional Top 8 blocks for manuscript readability.
- Mapbox background loading is a required Fig. 5a export step; a failed tile request stops the figure export rather than silently retaining a no-background map.
"""

LOG_PATH.write_text(textwrap.dedent(log_text), encoding="utf-8")
print(textwrap.dedent(log_text)[:1800])


# 06 Spatial Distribution Results Draft

## Figure inputs and method notes

- Source file: project main corpus CSV.
- Total deduplicated records: 9222
- Unique recipient institutions: 855
- English-name mapped institutions: 166
- English-name unmapped institutions retained as source recipient-organization names: 689
- Structured place fields for knowledge production, research object and beneficiary city contain no non-empty values in the current analytical dataset.
- Figure mode used: China province basemap and recipient-organization points reprojected to EPSG:3857 over a Web Mercator Mapbox WMTS background
- Region palette: seven low-saturation region colors derived from the Fig. 7 Sankey palette
- Mapbox WMTS background: Mapbox WMTS background; CRS=EPSG:3857; zoom=4; tiles=12/12
- Mapbox raster CRS: Web Mercator EPSG:3857; vector layers and points use the same projected CRS
- Mapbox cache write enabled: False
- Deterministic jitter: stable SHA-256 hash; radius capped at 50 km around 

In [10]:
# Run lightweight output QA.
# This cell reads existing Fig. 5/table/log files and validates the new-master-table scope; it does not write figures, tables, logs, or cache.
from PIL import Image
import json
import re

expected_files = [
    FIG_BASE.with_suffix(".svg"),
    FIG_BASE.with_suffix(".pdf"),
    FIG_BASE.with_suffix(".tiff"),
    FIG_BASE.with_suffix(".png"),
    TABLE_PATH,
    LOG_PATH,
]

qa_rows = []
for path in expected_files:
    qa_rows.append({
        "file": str(path.relative_to(ROOT)),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else 0,
    })
qa_df = pd.DataFrame(qa_rows)

with Image.open(FIG_BASE.with_suffix(".png")) as img:
    arr = np.asarray(img.convert("RGB"))
    png_width, png_height = img.size
    png_dpi = img.info.get("dpi")
    png_std = float(arr.std())
    png_white_share = float(np.mean(np.all(arr > 250, axis=2)))

with Image.open(FIG_BASE.with_suffix(".tiff")) as img:
    tiff_width, tiff_height = img.size
    tiff_dpi = img.info.get("dpi")
    tiff_arr = np.asarray(img.convert("RGB"))
    tiff_std = float(tiff_arr.std())
    tiff_white_share = float(np.mean(np.all(tiff_arr > 250, axis=2)))


def min_dpi(dpi):
    if dpi is None:
        return 0.0
    if isinstance(dpi, tuple):
        return min(float(value) for value in dpi)
    return float(dpi)


svg_text = FIG_BASE.with_suffix(".svg").read_text(encoding="utf-8", errors="ignore")
svg_text_lower = svg_text.lower()
log_text_check = LOG_PATH.read_text(encoding="utf-8", errors="ignore")
notebook_text_check = (ROOT / "code" / "06_spatial_distribution_analysis.ipynb").read_text(encoding="utf-8", errors="ignore")
svg_text_nodes = len(re.findall(r"<text", svg_text))
spatial_table_check = pd.read_csv(TABLE_PATH)

index_ids = (
    spatial_table_check.loc[spatial_table_check["section"].eq("institution_index"), "institution_id"]
    .dropna()
    .astype(int)
    .tolist()
)
expected_ids = list(range(1, n_institutions + 1))
points_ids = (
    spatial_table_check.loc[spatial_table_check["section"].eq("institution_points"), "institution_id"]
    .dropna()
    .astype(int)
    .sort_values()
    .tolist()
)
index_table_check = spatial_table_check.loc[spatial_table_check["section"].eq("institution_index")].copy()
index_sorted_ids = (
    index_table_check.sort_values(
        ["region_sort_rank", "province_sort_rank", "inferred_province", "institution_sort_name", "recipient_org"],
        kind="mergesort",
        na_position="last",
    )["institution_id"]
    .astype(int)
    .tolist()
)
used_regions = institution_points["region"].dropna().unique().tolist()
used_region_fill_colors = {region_fill_color(region).lower() for region in used_regions}
all_region_fill_colors = {region_fill_color(region).lower() for region in REGION_ORDER}
csv_text_check = TABLE_PATH.read_text(encoding="utf-8", errors="ignore")
mapbox_secret_token_pattern = r"pk\.[A-Za-z0-9._-]+"
text_cache_suffixes = {".xml", ".txt", ".json", ".md", ".csv", ".log", ".html", ".yml", ".yaml"}
cache_text_files = sorted(
    path for path in MAPBOX_CACHE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in text_cache_suffixes
) if MAPBOX_CACHE_DIR.exists() else []
token_scan_texts = {
    "output/figures/Fig5_spatial_distribution.svg": svg_text,
    "output/logs/06_spatial_text.md": log_text_check,
    "output/tables/06_spatial_distribution.csv": csv_text_check,
}
for path in cache_text_files:
    token_scan_texts[str(path.relative_to(ROOT))] = path.read_text(encoding="utf-8", errors="ignore")
mapbox_token_like_files = sorted(
    name for name, text in token_scan_texts.items()
    if re.search(mapbox_secret_token_pattern, text)
)

cjk_pattern = r"[一-鿿]"
source_field_coverage = spatial_table_check.loc[spatial_table_check["section"].eq("field_coverage")].copy()
record_count_from_table = int(source_field_coverage.loc[source_field_coverage["field"].eq("recipient_org"), "n_records"].iloc[0])
structured_counts = source_field_coverage.loc[source_field_coverage["field"].isin(STRUCTURED_PLACE_FIELDS), "n_records"].astype(int).tolist()
location_source_values = set(spatial_table_check.loc[spatial_table_check["section"].eq("location_source_distribution"), "place_name"].astype(str))
geo_audit_required_columns = {
    "source_type", "confidence", "needs_review", "source_url", "review_note",
    "geo_mapping_recipient_org_raw", "geo_mapping_city", "geo_mapping_province",
    "geo_mapping_lon_wgs84", "geo_mapping_lat_wgs84", "geo_mapping_source_type",
}
geo_row_sections = ["institution_base", "institution_points", "institution_index"]
geo_rows_check = spatial_table_check.loc[spatial_table_check["section"].isin(geo_row_sections)].copy()
inferred_city_records_check = int(source_field_coverage.loc[source_field_coverage["field"].eq("inferred_institution_city"), "n_records"].iloc[0])
inferred_province_records_check = int(source_field_coverage.loc[source_field_coverage["field"].eq("inferred_institution_province"), "n_records"].iloc[0])
forbidden_panel_c_title = "Coverage and inference " + "audit"
forbidden_coverage_note = "Coverage " + "note"
forbidden_point_note = "Each " + "point"
forbidden_panel_c_count_field = "n_" + "records"
forbidden_embedded_svg_fallback = "embedded SVG " + "fallback"
panel_c_full_names = region_top10["org_full"].astype(str) if "org_full" in region_top10.columns else region_top10["org_short"].astype(str)
disallowed_panel_c_abbrev_pattern = r"\b(?:" + "|".join(re.escape(term) for term in sorted(DISALLOWED_PANEL_C_ABBREVIATIONS, key=len, reverse=True)) + r")\b"
panel_c_disallowed_abbrev = sorted(set(term for term in DISALLOWED_PANEL_C_ABBREVIATIONS if re.search(r"\b" + re.escape(term) + r"\b", "\n".join(panel_c_full_names))))
region_top10_counts_check = region_top10.groupby("region").size().reindex(REGION_ORDER, fill_value=0)
region_top10_sorted_check = True
for region in REGION_ORDER:
    actual_ids = region_top10.loc[region_top10["region"].eq(region)].sort_values("regional_rank")["institution_id"].astype(int).tolist()
    expected_ids_region = (
        institution_index.loc[institution_index["region"].eq(region)]
        .assign(n_records=lambda df: pd.to_numeric(df["n_records"], errors="coerce").fillna(0).astype(int))
        .sort_values(["n_records", "institution_sort_name", "institution_id"], ascending=[False, True, True], kind="mergesort")
        .head(8)["institution_id"]
        .astype(int)
        .tolist()
    )
    region_top10_sorted_check = region_top10_sorted_check and actual_ids == expected_ids_region

qa_summary = {
    "figure_mode": figure_mode,
    "n_records": n_records,
    "expected_main_records": EXPECTED_MAIN_RECORDS,
    "n_institutions": n_institutions,
    "english_mapped_institutions": n_mapped_english_institutions,
    "english_unmapped_institutions": n_unmapped_english_institutions,
    "institution_index_rows_in_table": len(index_ids),
    "institution_points_rows_in_table": len(points_ids),
    "institution_index_ids_complete": index_ids == expected_ids,
    "institution_point_ids_subset": set(points_ids).issubset(set(expected_ids)),
    "institution_point_ids_complete": points_ids == expected_ids,
    "index_sorted_by_region_then_province": index_sorted_ids == index_ids,
    "city_top_count": len(city_top),
    "field_coverage_recipient_org_records": record_count_from_table,
    "structured_place_counts": structured_counts,
    "location_source_values": sorted(location_source_values),
    "no_unresolved_locations": "unresolved" not in location_source_values and not geo_rows_check["inferred_city"].isna().any(),
    "field_coverage_inferred_city_records": inferred_city_records_check,
    "field_coverage_inferred_province_records": inferred_province_records_check,
    "spatial_table_has_geo_audit_columns": geo_audit_required_columns.issubset(spatial_table_check.columns),
    "panel_c_csv_expected_orgs_all_in_english_map": bool(
        institution_index.loc[institution_index["recipient_org"].isin(PANEL_C_CSV_EXPECTED_ORGS), "recipient_org_en_mapped"].astype(bool).all()
    ),
    "panel_c_csv_expected_orgs_found_in_index": sorted(
        institution_index.loc[institution_index["recipient_org"].isin(PANEL_C_CSV_EXPECTED_ORGS), "recipient_org"].astype(str).tolist()
    ),
    "panel_c_csv_expected_orgs_override_fallback_used": sorted(
        region_top10.loc[
            region_top10["recipient_org"].isin(PANEL_C_CSV_EXPECTED_ORGS)
            & region_top10["org_full_source"].ne("recipient_org_english_name_map"),
            "recipient_org",
        ].astype(str).tolist()
    ) if "org_full_source" in region_top10.columns else [],
    "svg_has_panel_c_top_orgs": "Top 8 recipient organizations by province region" in svg_text,
    "svg_has_panel_c_audit": forbidden_panel_c_title in svg_text,
    "svg_has_coverage_note": forbidden_coverage_note in svg_text,
    "svg_has_each_point_note": forbidden_point_note in svg_text,
    "svg_has_panel_c_n_records_text": forbidden_panel_c_count_field in svg_text,
    "notebook_has_old_embedded_svg_fallback": forbidden_embedded_svg_fallback in notebook_text_check,
    "panel_c_full_names_have_cjk": bool(panel_c_full_names.str.contains(cjk_pattern, regex=True, na=False).any()),
    "panel_c_disallowed_abbreviations": panel_c_disallowed_abbrev,
    "svg_has_panel_c_disallowed_abbreviations": bool(re.search(disallowed_panel_c_abbrev_pattern, svg_text)),
    "svg_count_column_titles": len(re.findall(r">Count</text>", svg_text)),
    "panel_c_region_top8_counts": region_top10_counts_check.astype(int).to_dict(),
    "panel_c_has_8_orgs_per_region": bool(region_top10_counts_check.eq(8).all()),
    "panel_c_sorted_by_records_desc": bool(region_top10_sorted_check),
    "log_says_recipient_org_geography": "recipient-organization geography" in log_text_check,
    "log_says_not_study_area": "not as verified study-area" in log_text_check,
    "notebook_uses_new_main": MAIN_DATA_FILENAME in notebook_text_check,
    "svg_has_cjk_text": bool(re.search(cjk_pattern, svg_text)),
    "log_has_cjk_text": bool(re.search(cjk_pattern, log_text_check)),
    "mapbox_background_available": bool(mapbox_background_available),
    "mapbox_cache_write_enabled": bool(MAPBOX_CACHE_WRITE_ENABLED),
    "mapbox_tile_zoom": MAPBOX_TILE_ZOOM,
    "mapbox_background_status": scrub_mapbox_token(mapbox_background_status),
    "svg_has_mapbox_background_image": "<image" in svg_text_lower,
    "svg_has_gray_translucent_points": MAP_POINT_FILL.lower() in svg_text_lower,
    "svg_has_province_region_legend_title": "Province region" in svg_text,
    "svg_has_bold_province_region_title": bool(re.search(r"font: 700 [^>]*>Province region</text>", svg_text)),
    "svg_has_region_legend_labels": all(region in svg_text for region in REGION_ORDER),
    "svg_has_used_region_fills": used_region_fill_colors.issubset(set(re.findall(r"#[0-9a-f]{6}", svg_text_lower))),
    "svg_has_all_region_fills": all_region_fill_colors.issubset(set(re.findall(r"#[0-9a-f]{6}", svg_text_lower))),
    "spatial_table_has_region_columns": {"region", "region_color_group", "region_sort_rank"}.issubset(spatial_table_check.columns),
    "mapbox_cache_text_files": [str(path.relative_to(ROOT)) for path in cache_text_files],
    "mapbox_token_like_files": mapbox_token_like_files,
    "fig5_text_outputs_have_mapbox_token": bool(mapbox_token_like_files),
    "svg_has_unique_recipient_orgs_axis": "Unique recipient organizations" in svg_text,
    "png_width": png_width,
    "png_height": png_height,
    "png_dpi": png_dpi,
    "png_dpi_min": round(min_dpi(png_dpi), 2),
    "png_pixel_std": round(png_std, 2),
    "png_white_share": round(png_white_share, 4),
    "tiff_width": tiff_width,
    "tiff_height": tiff_height,
    "tiff_dpi": tiff_dpi,
    "tiff_dpi_min": round(min_dpi(tiff_dpi), 2),
    "tiff_pixel_std": round(tiff_std, 2),
    "tiff_white_share": round(tiff_white_share, 4),
    "svg_text_nodes": svg_text_nodes,
    "all_expected_files_exist": bool(qa_df["exists"].all()),
}

assert qa_summary["all_expected_files_exist"], qa_df
assert n_records == EXPECTED_MAIN_RECORDS, qa_summary
assert n_institutions == n_unique_institutions, qa_summary
assert n_mapped_english_institutions + n_unmapped_english_institutions == n_institutions, qa_summary
assert index_ids == expected_ids, qa_summary
assert points_ids == expected_ids, qa_summary
assert index_sorted_ids == index_ids, qa_summary
assert record_count_from_table == EXPECTED_MAIN_RECORDS, qa_summary
assert all(count == 0 for count in structured_counts), qa_summary
assert qa_summary["no_unresolved_locations"], qa_summary
assert inferred_city_records_check == EXPECTED_MAIN_RECORDS, qa_summary
assert inferred_province_records_check == EXPECTED_MAIN_RECORDS, qa_summary
assert qa_summary["spatial_table_has_geo_audit_columns"], qa_summary
assert qa_summary["panel_c_csv_expected_orgs_all_in_english_map"], qa_summary
assert not qa_summary["panel_c_csv_expected_orgs_override_fallback_used"], qa_summary
assert qa_summary["svg_has_panel_c_top_orgs"], qa_summary
assert not qa_summary["svg_has_panel_c_audit"], qa_summary
assert not qa_summary["svg_has_coverage_note"], qa_summary
assert not qa_summary["svg_has_each_point_note"], qa_summary
assert not qa_summary["svg_has_panel_c_n_records_text"], qa_summary
assert not qa_summary["notebook_has_old_embedded_svg_fallback"], qa_summary
assert not qa_summary["panel_c_full_names_have_cjk"], qa_summary
assert not qa_summary["panel_c_disallowed_abbreviations"], qa_summary
assert not qa_summary["svg_has_panel_c_disallowed_abbreviations"], qa_summary
assert qa_summary["svg_count_column_titles"] == 7, qa_summary
assert qa_summary["panel_c_has_8_orgs_per_region"], qa_summary
assert qa_summary["panel_c_sorted_by_records_desc"], qa_summary
assert qa_summary["log_says_recipient_org_geography"], qa_summary
assert qa_summary["log_says_not_study_area"], qa_summary
assert qa_summary["notebook_uses_new_main"], qa_summary
assert not re.search(cjk_pattern, svg_text), qa_summary
assert not re.search(cjk_pattern, log_text_check), qa_summary
assert qa_summary["spatial_table_has_region_columns"], qa_summary
assert not qa_summary["fig5_text_outputs_have_mapbox_token"], qa_summary
assert qa_summary["mapbox_background_available"], qa_summary
assert qa_summary["svg_has_mapbox_background_image"], qa_summary
assert qa_summary["svg_has_province_region_legend_title"], qa_summary
assert qa_summary["svg_has_bold_province_region_title"], qa_summary
assert "Unique recipient organizations" in svg_text, qa_summary
assert min_dpi(png_dpi) >= 590 and min_dpi(tiff_dpi) >= 590, qa_summary
assert png_width > 2500 and png_height > 2500 and png_std > 2 and png_white_share < 0.98, qa_summary
assert tiff_width > 2500 and tiff_height > 2500 and tiff_std > 2 and tiff_white_share < 0.98, qa_summary

qa_df, qa_summary


(                                            file  exists  size_kb
 0   output/figures/Fig5_spatial_distribution.svg    True   3462.9
 1   output/figures/Fig5_spatial_distribution.pdf    True   1335.6
 2  output/figures/Fig5_spatial_distribution.tiff    True  14875.5
 3   output/figures/Fig5_spatial_distribution.png    True   6700.9
 4      output/tables/06_spatial_distribution.csv    True   3301.6
 5                 output/logs/06_spatial_text.md    True     19.6,
 {'figure_mode': 'mapbox_wmts_background',
  'n_records': 9222,
  'expected_main_records': 9222,
  'n_institutions': 855,
  'english_mapped_institutions': 166,
  'english_unmapped_institutions': 689,
  'institution_index_rows_in_table': 855,
  'institution_points_rows_in_table': 855,
  'institution_index_ids_complete': True,
  'institution_point_ids_subset': True,
  'institution_point_ids_complete': True,
  'index_sorted_by_region_then_province': True,
  'city_top_count': 20,
  'field_coverage_recipient_org_records': 9222,
 